# FTWP Chapter 2 Capstone — Replication and Arbitrage-Free Pricing

## Exact Replication, Market Completeness, and Numerical Robustness

### Financial Question

If a contingent claim has a state-dependent payoff, can a portfolio of traded
assets be constructed that reproduces that payoff exactly in every possible
future state?

If exact replication is possible, what does the replicating portfolio cost
today, and why does that cost determine the claim's no-arbitrage price?

This notebook develops that logic in a two-state economy and then examines what
changes when the traded-asset payoff matrix becomes singular or nearly
singular.

### Source and Scope

This notebook is a worked implementation and analysis of the Chapter 2
capstone in *Financial Theory with Python — Exercises*, by Yves J. Hilpisch,
The Python Quants GmbH, page 10.

The official task has four main components:

1. Build a reusable pricing function that accepts a market payoff matrix
   $M$, current traded-asset prices $M_0$, and a contingent-claim payoff
   vector $C_1$, and returns the replication portfolio and claim price.

2. Add numerical checks that detect a singular payoff matrix and verify the
   norm of the replication residual.

3. Stress-test the pricing rule with 100 randomly generated claims and verify
   linearity of pricing.

4. Investigate a nearly singular payoff matrix and compare
   `np.linalg.solve` with `np.linalg.lstsq`.

### Organization

The notebook proceeds through the following sections:

1. Define the two-date, two-state market.
2. Construct the traded-asset payoff matrix.
3. Derive the replication equations by hand.
4. Solve and verify the replicating portfolio numerically.
5. Derive the claim's no-arbitrage price from replication.
6. Build a reusable replication-pricing function.
7. Interpret singularity and test controlled failure.
8. Verify linearity of replication pricing.
9. Run a reproducible 100-claim stress test.
10. Analyze a nearly singular market and compare numerical solvers.
11. Summarize the financial conclusions.
12. Perform reproducibility and validation checks.

### Interpretation Boundary

The model is a stylized two-date, two-state economy.

Within this model, exact replication means matching the claim payoff in every
modeled future state.

The notebook distinguishes carefully between

- current asset prices at $t=0$;
- future state-dependent payoffs at $t=1$;
- portfolio holdings;
- replication residuals;
- and the arbitrage-free price implied by exact replication.

Numerical solvability, economic interpretation, and numerical robustness will
be treated as related but distinct questions.

In [1]:
import numpy as np

print("NumPy is ready.")

NumPy is ready.


## Part 1 — Dates, States, and the Two Traded Assets

### Financial Setup

The model has two dates:

- $t=0$: today, when traded assets have observable market prices;
- $t=1$: the future date, when one of two possible states is realized.

The future states are ordered as

$$
(\text{Up},\text{Down}).
$$

This ordering must remain consistent across every payoff vector, matrix row,
claim vector, and numerical calculation in the notebook.

---

### 1. Risk-Free Bond

The bond costs

$$
B_0=10
$$

at $t=0$.

At $t=1$, it pays

$$
B_1=
\begin{pmatrix}
11\\
11
\end{pmatrix}.
$$

The payoff is the same in both future states.

Therefore, within this stylized model, the bond is risk-free with respect to
the modeled state uncertainty.

Its one-period rate of return is

$$
i
=
\frac{11-10}{10}
=
0.10.
$$

Thus,

$$
\boxed{i=10\%}.
$$

"Risk-free" does not mean that the bond produces no return.

It means that its future payoff does not depend on whether the Up or Down state
occurs.

---

### 2. Risky Stock

The stock also costs

$$
S_0=10
$$

today.

Its future payoff depends on the realized state:

$$
S_1=
\begin{pmatrix}
20\\
5
\end{pmatrix}.
$$

Thus,

$$
S_1^u=20,
\qquad
S_1^d=5.
$$

Unlike the bond, the stock delivers different future values across the two
states.

That state dependence is the source of risk in this discrete model.

---

### 3. Target Contingent Claim

The target claim is a European call option written on the stock with strike

$$
K=15.
$$

Its maturity payoff is

$$
C_1
=
\max(S_1-K,0)
$$

state by state.

In the Up state,

$$
C_1^u
=
\max(20-15,0)
=
5.
$$

In the Down state,

$$
C_1^d
=
\max(5-15,0)
=
0.
$$

Therefore,

$$
\boxed{
C_1=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
}
$$

The claim is a contingent claim because its future cash flow depends on which
state occurs.

---

### 4. Prices Today Versus Payoffs Tomorrow

A central distinction in the replication problem is between current prices and
future payoffs.

At $t=0$,

$$
B_0=10,
\qquad
S_0=10.
$$

These are market prices today.

At $t=1$,

$$
B_1=(11,11)
$$

and

$$
S_1=(20,5)
$$

are state-dependent future payoffs.

The claim payoff

$$
C_1=(5,0)
$$

also belongs to $t=1$.

The replication problem will ask whether holdings of the bond and stock can
produce exactly this future claim payoff in every state.

Only after that portfolio has been identified will its $t=0$ cost be used to
determine the claim's arbitrage-free price.

In [2]:
states = ["Up", "Down"]

B0 = 10.0
B1 = np.array([11.0, 11.0])

S0 = 10.0
S1 = np.array([20.0, 5.0])

K = 15.0
C1 = np.maximum(S1 - K, 0.0)

risk_free_rate = B1[0] / B0 - 1.0

print("States:", states)
print()
print("Bond price today B0:", B0)
print("Bond payoff B1:", B1)
print("One-period risk-free rate:", risk_free_rate)

print()
print("Stock price today S0:", S0)
print("Stock payoff S1:", S1)

print()
print("Call strike K:", K)
print("Claim payoff C1:", C1)

print()
print("Bond payoff shape:", B1.shape)
print("Stock payoff shape:", S1.shape)
print("Claim payoff shape:", C1.shape)

States: ['Up', 'Down']

Bond price today B0: 10.0
Bond payoff B1: [11. 11.]
One-period risk-free rate: 0.10000000000000009

Stock price today S0: 10.0
Stock payoff S1: [20.  5.]

Call strike K: 15.0
Claim payoff C1: [5. 0.]

Bond payoff shape: (2,)
Stock payoff shape: (2,)
Claim payoff shape: (2,)


### Interpretation of the Market Setup

The Python output reproduces the two-date, two-state economy exactly.

The current bond price is

$$
B_0=10,
$$

and its future payoff vector is

$$
B_1=
\begin{pmatrix}
11\\
11
\end{pmatrix}.
$$

Because the bond pays $11$ in both future states, its payoff does not depend on
which modeled state occurs.

Its one-period return is therefore

$$
i
=
\frac{11}{10}-1
=
0.10.
$$

Thus,

$$
\boxed{
i=10\%.
}
$$

Python displays this as approximately

$$
0.10000000000000009,
$$

which is a floating-point representation effect rather than a financially
meaningful difference from $0.10$.

---

#### 1. The Bond Is Risk-Free Within the Model

The bond payoff is

$$
B_1=(11,11).
$$

Its Up-state and Down-state payoffs are identical:

$$
B_1^u=B_1^d=11.
$$

Therefore the bond has no payoff uncertainty across the two modeled states.

This is what "risk-free" means in the present discrete model.

It does not mean that the bond earns no return.

The bond costs $10$ today and pays $11$ tomorrow, so it earns a certain
one-period return of $10\%$.

---

#### 2. The Stock Is State-Dependent

The stock costs

$$
S_0=10
$$

today and has future payoff vector

$$
S_1=
\begin{pmatrix}
20\\
5
\end{pmatrix}.
$$

Thus,

$$
S_1^u=20
$$

in the Up state and

$$
S_1^d=5
$$

in the Down state.

Because these payoffs differ across states, the stock is risky with respect to
the uncertainty modeled here.

---

#### 3. The Call Is a State-Contingent Claim

The strike is

$$
K=15.
$$

The call payoff rule is

$$
C_1=\max(S_1-K,0).
$$

State by state,

$$
C_1^u
=
\max(20-15,0)
=
5
$$

and

$$
C_1^d
=
\max(5-15,0)
=
0.
$$

Therefore,

$$
\boxed{
C_1=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
}
$$

The claim payoff depends on the future state, which is why it is a contingent
claim.

---

#### 4. All Future Payoff Objects Live in the Same State Space

Python reports

$$
B_1.\mathrm{shape}
=
S_1.\mathrm{shape}
=
C_1.\mathrm{shape}
=
(2,).
$$

Each object therefore contains one payoff for each of the two future states.

Mathematically,

$$
B_1,S_1,C_1\in\mathbb{R}^2.
$$

This common state dimension is essential for replication.

A portfolio can only be compared with the claim state by state if the traded
asset payoff vectors and the claim payoff vector refer to the same states in
the same order.

---

#### 5. State Ordering Must Remain Consistent

The notebook uses

$$
(\text{Up},\text{Down})
$$

throughout.

Therefore the first component of every future payoff vector corresponds to the
Up state and the second component corresponds to the Down state.

For example,

$$
S_1=(20,5)
$$

means

$$
(S_1^u,S_1^d)=(20,5),
$$

while

$$
C_1=(5,0)
$$

means

$$
(C_1^u,C_1^d)=(5,0).
$$

Python itself does not know the financial meaning of these positions. The
state interpretation comes from the model and must be preserved by the
implementation.

---

#### 6. Prices and Payoffs Are Different Objects

The quantities

$$
B_0
\quad\text{and}\quad
S_0
$$

are prices at

$$
t=0.
$$

By contrast,

$$
B_1,\quad S_1,\quad C_1
$$

are payoffs or values at

$$
t=1.
$$

This distinction matters because the replication problem has two stages.

First, the future payoff question is:

$$
\boxed{
\text{Can a bond-stock portfolio reproduce }C_1
\text{ in every future state?}
}
$$

Second, if such a portfolio exists, the pricing question is:

$$
\boxed{
\text{What does that replicating portfolio cost at }t=0?
}
$$

The future payoff equations determine the hedge holdings.

The current asset prices then determine the cost of that hedge.

---

#### 7. Transition to the Market Payoff Matrix

The next step is to organize the bond and stock payoff vectors into a single
market payoff matrix.

That matrix will encode the future cash-flow opportunities generated by the
two traded assets.

The replication question can then be written compactly as

$$
M\phi=C_1,
$$

where

$$
\phi
=
\begin{pmatrix}
\phi_B\\
\phi_S
\end{pmatrix}
$$

contains the bond and stock holdings.

Before solving that system numerically, the matrix structure and its financial
meaning will be established explicitly.

## Part 2 — Constructing the Market Payoff Matrix

The two traded assets generate future cash flows across the same two future
states.

Their payoff vectors are

$$
B_1=
\begin{pmatrix}
11\\
11
\end{pmatrix}
$$

for the bond and

$$
S_1=
\begin{pmatrix}
20\\
5
\end{pmatrix}
$$

for the stock.

To analyze replication systematically, these payoff vectors are combined into
the market payoff matrix

$$
M=
\begin{pmatrix}
11 & 20\\
11 & 5
\end{pmatrix}.
$$

The matrix convention used throughout the notebook is

$$
\boxed{
\text{rows}=\text{future states},
\qquad
\text{columns}=\text{traded assets}.
}
$$

Therefore,

- the first row corresponds to the Up state;
- the second row corresponds to the Down state;
- the first column contains the bond payoffs;
- the second column contains the stock payoffs.

---

### 1. Portfolio Holdings

Let

$$
\phi=
\begin{pmatrix}
\phi_B\\
\phi_S
\end{pmatrix}
$$

denote a portfolio containing

- $\phi_B$ units of the bond;
- $\phi_S$ units of the stock.

These quantities are asset holdings, not automatically percentages or
portfolio weights.

The portfolio's future payoff vector is

$$
M\phi.
$$

Explicitly,

$$
M\phi
=
\begin{pmatrix}
11 & 20\\
11 & 5
\end{pmatrix}
\begin{pmatrix}
\phi_B\\
\phi_S
\end{pmatrix}
=
\begin{pmatrix}
11\phi_B+20\phi_S\\
11\phi_B+5\phi_S
\end{pmatrix}.
$$

The first component is the portfolio payoff in the Up state and the second is
the portfolio payoff in the Down state.

---

### 2. Replication as a Linear System

The target call claim is

$$
C_1=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
$$

Exact replication requires the traded-asset portfolio to match the claim in
every state:

$$
\boxed{
M\phi=C_1.
}
$$

Therefore,

$$
\begin{pmatrix}
11 & 20\\
11 & 5
\end{pmatrix}
\begin{pmatrix}
\phi_B\\
\phi_S
\end{pmatrix}
=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
$$

This matrix equation represents two state-by-state equations:

$$
11\phi_B+20\phi_S=5
$$

and

$$
11\phi_B+5\phi_S=0.
$$

The next section will solve these equations by hand before using a numerical
linear-algebra routine.

Before solving, however, it is useful to determine whether the payoff matrix
has enough independent payoff directions to span the two-state payoff space.

---

### 3. Determinant and Linear Independence

For a $2\times2$ matrix

$$
M=
\begin{pmatrix}
a&b\\
c&d
\end{pmatrix},
$$

the determinant is

$$
\det(M)=ad-bc.
$$

For this market,

$$
\det(M)
=
11(5)-20(11).
$$

Therefore,

$$
\det(M)
=
55-220
=
-165.
$$

Hence,

$$
\boxed{
\det(M)=-165\neq0.
}
$$

A nonzero determinant means that the two columns of $M$ are linearly
independent.

Financially, the bond and stock provide two independent future payoff
directions.

---

### 4. Rank and Completeness

Because the two columns are linearly independent,

$$
\operatorname{rank}(M)=2.
$$

The entire two-state contingent-claim space is

$$
\mathbb{R}^2.
$$

Since the traded payoffs span two independent directions in this
two-dimensional space,

$$
\boxed{
\operatorname{col}(M)=\mathbb{R}^2.
}
$$

Thus, under the model's unrestricted portfolio-holding assumption, every
two-state payoff vector can be represented by some bond-stock portfolio.

This is the meaning of market completeness in this two-state example.

Therefore,

$$
\boxed{
\text{two independent traded payoff vectors}
\Longrightarrow
\text{complete two-state market}.
}
$$

In particular, the call payoff

$$
C_1=(5,0)
$$

must be attainable.

---

### 5. Current Asset Price Vector

The corresponding current prices are collected as

$$
M_0=
\begin{pmatrix}
B_0\\
S_0
\end{pmatrix}
=
\begin{pmatrix}
10\\
10
\end{pmatrix}.
$$

The future payoff matrix $M$ will determine which holdings replicate the
claim.

The current price vector $M_0$ will later determine what those holdings cost
today.

Keeping these objects separate preserves the distinction between

$$
\boxed{
\text{future replication}
}
$$

and

$$
\boxed{
\text{current valuation}.
}
$$

In [3]:
M = np.column_stack((B1, S1))
M0 = np.array([B0, S0])

det_M = np.linalg.det(M)
rank_M = np.linalg.matrix_rank(M)

print("Market payoff matrix M:")
print(M)

print()
print("Current traded-asset prices M0:", M0)
print("Target claim payoff C1:", C1)

print()
print("Matrix shape:", M.shape)
print("Determinant of M:", det_M)
print("Rank of M:", rank_M)
print("Matrix is invertible:", not np.isclose(det_M, 0.0))
print("Two-state market is complete:", rank_M == 2)

Market payoff matrix M:
[[11. 20.]
 [11.  5.]]

Current traded-asset prices M0: [10. 10.]
Target claim payoff C1: [5. 0.]

Matrix shape: (2, 2)
Determinant of M: -165.00000000000009
Rank of M: 2
Matrix is invertible: True
Two-state market is complete: True


### Interpretation of the Market Payoff Matrix

The Python calculation constructs the market payoff matrix

$$
M=
\begin{pmatrix}
11&20\\
11&5
\end{pmatrix},
$$

the current traded-asset price vector

$$
M_0=
\begin{pmatrix}
10\\
10
\end{pmatrix},
$$

and the target claim payoff

$$
C_1=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
$$

The matrix has shape

$$
(2,2),
$$

because the model contains two future states and two traded assets.

---

#### 1. Financial Meaning of the Matrix Entries

The first column of $M$ is the bond payoff vector:

$$
\begin{pmatrix}
11\\
11
\end{pmatrix}.
$$

The second column is the stock payoff vector:

$$
\begin{pmatrix}
20\\
5
\end{pmatrix}.
$$

The first row corresponds to the Up state:

$$
(11,20),
$$

meaning that in that state the bond pays $11$ and the stock is worth $20$.

The second row corresponds to the Down state:

$$
(11,5),
$$

meaning that in that state the bond again pays $11$ while the stock is worth
$5$.

Thus,

$$
\boxed{
\text{rows encode states and columns encode traded assets}.
}
$$

---

#### 2. Determinant and Invertibility

Python returns

$$
\det(M)
\approx
-165.00000000000009.
$$

The exact determinant is

$$
\det(M)
=
11(5)-11(20)
=
55-220
=
-165.
$$

Therefore,

$$
\boxed{
\det(M)=-165\neq0.
}
$$

The tiny discrepancy between the exact value and the printed decimal is only a
floating-point representation effect.

Because the determinant is nonzero, the matrix is invertible.

Hence,

$$
\boxed{
M^{-1}
\text{ exists}.
}
$$

---

#### 3. Linear Independence of the Traded Payoffs

A nonzero determinant also means that the two columns of $M$ are linearly
independent.

Financially, the bond and stock do not generate redundant future payoff
patterns.

The bond provides the direction

$$
(11,11),
$$

while the stock provides

$$
(20,5).
$$

Neither payoff vector is a scalar multiple of the other.

Therefore, they span two independent directions in the two-dimensional future
payoff space.

---

#### 4. Rank and Completeness

Python confirms

$$
\operatorname{rank}(M)=2.
$$

The entire contingent-claim payoff space in this model is

$$
\mathbb{R}^2.
$$

Because the two traded payoff vectors span a two-dimensional space,

$$
\operatorname{col}(M)=\mathbb{R}^2.
$$

Therefore every two-state payoff vector can be represented by a bond-stock
portfolio, provided unrestricted holdings are allowed.

Hence,

$$
\boxed{
\text{the market is complete in this two-state model}.
}
$$

This means that the particular claim

$$
C_1=(5,0)
$$

is not merely expected to be attainable: the market structure guarantees that
a unique replicating portfolio exists.

---

#### 5. Why Invertibility Implies a Unique Replicating Portfolio

The replication problem is

$$
M\phi=C_1.
$$

Because $M$ is invertible,

$$
\phi=M^{-1}C_1
$$

exists and is unique.

Thus, there cannot be two different bond-stock portfolios producing the same
claim payoff in this complete two-state market.

The market therefore determines a unique pair of holdings

$$
\phi_B,\phi_S
$$

for the selected claim.

---

#### 6. Future Replication Versus Current Cost

The payoff matrix

$$
M
$$

contains future cash flows.

The vector

$$
M_0=(10,10)
$$

contains current prices.

These objects play different roles.

First,

$$
M\phi=C_1
$$

determines the portfolio holdings required to reproduce the claim at $t=1$.

Only after those holdings are known do we calculate the portfolio's current
cost:

$$
\phi^\top M_0.
$$

Thus,

$$
\boxed{
\text{future payoffs determine the hedge;}
}
$$

$$
\boxed{
\text{current prices determine the cost of that hedge.}
}
$$

This separation is central to replication-based pricing.

---

#### 7. Why No State Probabilities Are Needed Yet

The replication equation requires the portfolio payoff to equal the claim
payoff in **every possible state**.

Therefore, the hedge does not depend on how likely the Up or Down state is.

If

$$
M\phi=C_1,
$$

then the equality holds state by state regardless of the physical probability
assigned to those states.

Probabilities may become useful for other purposes, but they are not required
to solve this exact replication problem.

---

#### 8. Transition

The matrix diagnostics establish that

$$
M
$$

is invertible and that the claim can be uniquely replicated.

The next step is therefore to solve

$$
M\phi=C_1
$$

explicitly.

Rather than asking Python to produce the holdings immediately, the replication
equations will first be solved by hand so that the financial meaning of the
bond and stock positions is clear before the numerical linear-algebra result is
introduced.

## Part 3 — Deriving the Replicating Portfolio by Hand

The market diagnostics established that the payoff matrix

$$
M=
\begin{pmatrix}
11&20\\
11&5
\end{pmatrix}
$$

is invertible.

Therefore, the claim

$$
C_1=
\begin{pmatrix}
5\\
0
\end{pmatrix}
$$

has a unique replicating portfolio.

Let

$$
\phi=
\begin{pmatrix}
\phi_B\\
\phi_S
\end{pmatrix},
$$

where

- $\phi_B$ is the number of bond units held;
- $\phi_S$ is the number of stock units held.

Exact replication requires

$$
M\phi=C_1.
$$

Therefore,

$$
\begin{pmatrix}
11&20\\
11&5
\end{pmatrix}
\begin{pmatrix}
\phi_B\\
\phi_S
\end{pmatrix}
=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
$$

This produces one equation for each future state.

---

### 1. Up-State Replication Equation

In the Up state, the bond pays $11$, the stock pays $20$, and the call must pay
$5$.

Therefore,

$$
11\phi_B+20\phi_S=5.
$$

---

### 2. Down-State Replication Equation

In the Down state, the bond pays $11$, the stock pays $5$, and the call must pay
$0$.

Therefore,

$$
11\phi_B+5\phi_S=0.
$$

The complete system is

$$
\boxed{
\begin{aligned}
11\phi_B+20\phi_S &= 5,\\
11\phi_B+5\phi_S &= 0.
\end{aligned}
}
$$

---

### 3. Solve for the Stock Holding

Subtract the Down-state equation from the Up-state equation:

$$
(11\phi_B+20\phi_S)
-
(11\phi_B+5\phi_S)
=
5-0.
$$

The bond terms cancel:

$$
15\phi_S=5.
$$

Therefore,

$$
\boxed{
\phi_S=\frac13.
}
$$

The replicating strategy must hold one-third of a stock unit.

---

### 4. Solve for the Bond Holding

Substitute

$$
\phi_S=\frac13
$$

into the Down-state equation:

$$
11\phi_B
+
5\left(\frac13\right)
=
0.
$$

Thus,

$$
11\phi_B+\frac53=0.
$$

Therefore,

$$
11\phi_B=-\frac53,
$$

so

$$
\boxed{
\phi_B=-\frac5{33}.
}
$$

The negative bond holding has an economic interpretation: the replicating
strategy takes a short position in the bond, or equivalently borrows through
the bond position.

---

### 5. Replicating Portfolio

The unique replicating portfolio is therefore

$$
\boxed{
\phi=
\begin{pmatrix}
-\frac5{33}\\
\frac13
\end{pmatrix}.
}
$$

In `(bond, stock)` order,

$$
\boxed{
(\phi_B,\phi_S)
=
\left(
-\frac5{33},
\frac13
\right).
}
$$

Numerically,

$$
\phi
\approx
\begin{pmatrix}
-0.15152\\
0.33333
\end{pmatrix}.
$$

---

### 6. Manual State-by-State Verification

A replication portfolio is not established merely by solving for two numbers.

The holdings must reproduce the claim payoff in every future state.

#### Up State

The bond contribution is

$$
11\left(-\frac5{33}\right)
=
-\frac53.
$$

The stock contribution is

$$
20\left(\frac13\right)
=
\frac{20}{3}.
$$

Therefore,

$$
-\frac53+\frac{20}{3}
=
\frac{15}{3}
=
5.
$$

Thus,

$$
\boxed{
V_1^u=5=C_1^u.
}
$$

#### Down State

The bond contribution is again

$$
11\left(-\frac5{33}\right)
=
-\frac53.
$$

The stock contribution is

$$
5\left(\frac13\right)
=
\frac53.
$$

Therefore,

$$
-\frac53+\frac53=0.
$$

Thus,

$$
\boxed{
V_1^d=0=C_1^d.
}
$$

Combining the two states,

$$
\boxed{
M\phi
=
\begin{pmatrix}
5\\
0
\end{pmatrix}
=
C_1.
}
$$

The call is therefore replicated exactly by the bond-stock portfolio.

---

### 7. Financial Interpretation of the Holdings

The replicating strategy contains

$$
\phi_S=\frac13>0,
$$

so it holds a long position in the stock.

It also contains

$$
\phi_B=-\frac5{33}<0,
$$

so it takes a negative position in the bond.

The negative bond position helps finance the positive stock holding while also
shaping the portfolio's state-dependent payoff.

These positions are quantities of traded assets.

They should not be interpreted automatically as percentages or portfolio
weights.

---

### 8. Why Probabilities Still Do Not Enter

No probability vector was needed to derive the hedge.

The reason is that replication requires

$$
M\phi=C_1
$$

state by state.

The portfolio must reproduce the call payoff whether the Up state is very
likely, very unlikely, or assigned any other admissible physical probability.

Thus,

$$
\boxed{
\text{exact replication depends on payoff matching, not state likelihoods}.
}
$$

The next step is to reproduce the hand-derived portfolio with numerical linear
algebra and verify the residual explicitly.

## Part 4 — Numerical Solution and Replication Verification

The hand derivation produced

$$
\phi=
\begin{pmatrix}
-\frac5{33}\\
\frac13
\end{pmatrix}.
$$

The same linear system can be solved computationally using

`np.linalg.solve(M, C1)`.

Because $M$ is square and invertible, `np.linalg.solve` returns the unique
vector $\phi$ satisfying

$$
M\phi=C_1.
$$

The numerical result will be checked in three ways:

1. compare the computed holdings with the hand-derived holdings;
2. reconstruct the portfolio payoff with $M\phi$;
3. calculate the replication residual

$$
e=M\phi-C_1.
$$

For exact replication, the residual should be zero apart from possible
floating-point rounding error.

In [4]:
manual_phi = np.array([
    -5.0 / 33.0,
    1.0 / 3.0,
])

phi = np.linalg.solve(M, C1)

phi_B, phi_S = phi

bond_payoff_contribution = phi_B * B1
stock_payoff_contribution = phi_S * S1

replicated_payoff = M @ phi
replication_residual = replicated_payoff - C1
residual_norm = np.linalg.norm(replication_residual)

print("Hand-derived holdings:", manual_phi)
print("Numerical holdings:", phi)

print()
print("Bond holding phi_B:", phi_B)
print("Stock holding phi_S:", phi_S)

print()
print("Bond payoff contribution:", bond_payoff_contribution)
print("Stock payoff contribution:", stock_payoff_contribution)

print()
print("Replicated payoff:", replicated_payoff)
print("Target claim payoff:", C1)
print("Replication residual:", replication_residual)
print("Residual norm:", residual_norm)

print()
print(
    "Numerical holdings match hand derivation:",
    np.allclose(phi, manual_phi),
)
print(
    "Portfolio replicates claim:",
    np.allclose(replicated_payoff, C1),
)

Hand-derived holdings: [-0.15151515  0.33333333]
Numerical holdings: [-0.15151515  0.33333333]

Bond holding phi_B: -0.15151515151515146
Stock holding phi_S: 0.3333333333333333

Bond payoff contribution: [-1.66666667 -1.66666667]
Stock payoff contribution: [6.66666667 1.66666667]

Replicated payoff: [5.00000000e+00 4.16333634e-16]
Target claim payoff: [5. 0.]
Replication residual: [0.00000000e+00 4.16333634e-16]
Residual norm: 4.163336342344337e-16

Numerical holdings match hand derivation: True
Portfolio replicates claim: True


### Derivation and Interpretation of the Numerical Replication

The numerical solution returns

$$
\phi
\approx
\begin{pmatrix}
-0.15151515\\
0.33333333
\end{pmatrix},
$$

which matches the exact hand-derived portfolio

$$
\boxed{
\phi
=
\begin{pmatrix}
-\frac5{33}\\
\frac13
\end{pmatrix}.
}
$$

Python also confirms this agreement with

`Numerical holdings match hand derivation: True`.

---

#### 1. Bond and Stock Contributions

The bond position is

$$
\phi_B=-\frac5{33},
$$

so its future payoff contribution is

$$
\phi_B B_1
=
-\frac5{33}
\begin{pmatrix}
11\\
11
\end{pmatrix}
=
\begin{pmatrix}
-\frac53\\
-\frac53
\end{pmatrix}.
$$

Numerically,

$$
\phi_B B_1
\approx
(-1.66667,-1.66667).
$$

The stock position is

$$
\phi_S=\frac13,
$$

so its payoff contribution is

$$
\phi_S S_1
=
\frac13
\begin{pmatrix}
20\\
5
\end{pmatrix}
=
\begin{pmatrix}
\frac{20}{3}\\
\frac53
\end{pmatrix}.
$$

Numerically,

$$
\phi_S S_1
\approx
(6.66667,1.66667).
$$

---

#### 2. State-by-State Replication

Adding the bond and stock contributions gives

$$
M\phi
=
\begin{pmatrix}
-\frac53+\frac{20}{3}\\
-\frac53+\frac53
\end{pmatrix}
=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
$$

Therefore,

$$
\boxed{
M\phi=C_1.
}
$$

The portfolio reproduces the claim exactly in every modeled state.

---

#### 3. Replication Residual

The replication residual is defined as

$$
e=M\phi-C_1.
$$

Python returns

$$
e
\approx
\begin{pmatrix}
0\\
4.16\times10^{-16}
\end{pmatrix}.
$$

The Euclidean norm is

$$
\|e\|_2
\approx
4.16\times10^{-16}.
$$

This quantity is effectively zero at machine precision.

The nonzero printed value does not represent a meaningful financial mismatch.
It arises from finite-precision floating-point arithmetic.

Thus,

$$
\boxed{
\|e\|_2\approx0
}
$$

for practical numerical purposes.

---

#### 4. Why `np.allclose` Is Used

In exact mathematics, successful replication means

$$
M\phi=C_1
$$

with zero residual.

In floating-point computation, algebraically exact quantities may be stored as
very small approximations.

Therefore, requiring

`replicated_payoff == C1`

component by component would be unnecessarily strict.

Instead,

`np.allclose(replicated_payoff, C1)`

checks whether the numerical values agree within a small tolerance.

Python returns

`Portfolio replicates claim: True`.

This is the appropriate numerical interpretation of exact replication in a
floating-point environment.

---

#### 5. Economic Meaning of the Negative Bond Holding

The bond holding is

$$
\phi_B=-\frac5{33}.
$$

A negative asset holding represents a short position.

In this simple one-period model, the negative bond position can be interpreted
as borrowing.

The strategy uses that borrowing together with the positive stock position to
produce the claim's desired state-dependent payoff.

Thus, the replicating portfolio is economically meaningful:

$$
\boxed{
\text{borrow through the bond}
+
\text{hold stock}
=
\text{replicate the call}.
}
$$

---

#### 6. Why Exact Replication Is Powerful

The portfolio and the claim produce the same future payoff vector:

$$
M\phi=C_1.
$$

Therefore, within the assumptions of the model, the two positions are
financially equivalent at maturity.

This observation is the bridge from hedging to pricing.

If two positions have identical future cash flows in every possible state but
different prices today, the cheaper one could be purchased and the more
expensive one sold, creating an arbitrage opportunity.

Therefore, absence of arbitrage requires the claim's current price to equal the
current cost of its replicating portfolio.

The next section derives that cost explicitly.

## Part 5 — Arbitrage-Free Price from Replication

The previous section established that the portfolio

$$
\phi=
\begin{pmatrix}
-\frac5{33}\\
\frac13
\end{pmatrix}
$$

replicates the call payoff exactly:

$$
M\phi=C_1.
$$

Because the portfolio and the claim have identical future cash flows in every
modeled state, the no-arbitrage principle links their values today.

The claim's arbitrage-free price must equal the current cost of the replicating
portfolio.

---

### 1. Current Cost of the Replicating Portfolio

The current traded-asset price vector is

$$
M_0=
\begin{pmatrix}
B_0\\
S_0
\end{pmatrix}
=
\begin{pmatrix}
10\\
10
\end{pmatrix}.
$$

The time-$0$ cost of portfolio $\phi$ is

$$
C_0
=
\phi^\top M_0.
$$

Equivalently,

$$
C_0
=
\phi_B B_0+\phi_S S_0.
$$

Substituting the exact holdings,

$$
C_0
=
\left(-\frac5{33}\right)(10)
+
\left(\frac13\right)(10).
$$

Therefore,

$$
C_0
=
-\frac{50}{33}
+
\frac{10}{3}.
$$

Using a common denominator,

$$
\frac{10}{3}
=
\frac{110}{33},
$$

so

$$
C_0
=
-\frac{50}{33}
+
\frac{110}{33}
=
\frac{60}{33}.
$$

Hence,

$$
\boxed{
C_0
=
\frac{20}{11}
\approx
1.81818.
}
$$

---

### 2. Interpreting the Cost Components

The bond position is negative:

$$
\phi_B=-\frac5{33}.
$$

Its current contribution to the portfolio cost is

$$
\phi_BB_0
=
-\frac5{33}(10)
=
-\frac{50}{33}
\approx
-1.51515.
$$

The stock position is positive:

$$
\phi_S=\frac13.
$$

Its current cost is

$$
\phi_SS_0
=
\frac13(10)
=
\frac{10}{3}
\approx
3.33333.
$$

Thus,

$$
-1.51515+3.33333
\approx
1.81818.
$$

The negative bond position reduces the cash required to finance the positive
stock position.

---

### 3. Why Replication Determines the Price

The claim and the replicating portfolio have identical future payoffs:

$$
M\phi=C_1.
$$

Suppose instead that the claim traded today at some market price

$$
\widetilde C_0
$$

different from the replication cost

$$
C_0=\frac{20}{11}.
$$

Then the claim and the replicating portfolio would have the same future cash
flows but different current prices.

That difference would create an arbitrage opportunity.

---

### 4. If the Claim Is Too Expensive

Suppose

$$
\widetilde C_0>C_0.
$$

Then the claim is more expensive than the portfolio that reproduces it.

One could:

1. sell the expensive claim;
2. buy the cheaper replicating portfolio.

The initial net cash flow would be

$$
\widetilde C_0-C_0>0.
$$

At maturity, the replicating portfolio produces exactly the cash flow required
to satisfy the short claim position.

Therefore, the future net payoff is zero in every state while the initial cash
flow is positive.

That is an arbitrage.

---

### 5. If the Claim Is Too Cheap

Suppose instead

$$
\widetilde C_0<C_0.
$$

Then the claim is cheaper than the portfolio that replicates it.

One could:

1. buy the cheap claim;
2. short the more expensive replicating portfolio.

The initial net cash flow would be

$$
C_0-\widetilde C_0>0.
$$

At maturity, the claim payoff exactly offsets the liability from the short
replicating portfolio.

Again, the future net payoff is zero in every state while the initial cash flow
is positive.

That is also an arbitrage.

---

### 6. No-Arbitrage Conclusion

Both possible mispricings lead to arbitrage.

Therefore, absence of arbitrage requires

$$
\boxed{
\widetilde C_0=C_0.
}
$$

Hence the unique arbitrage-free price of the replicated call is

$$
\boxed{
C_0=\frac{20}{11}\approx1.81818.
}
$$

This pricing result is not obtained by averaging future payoffs under the
physical probabilities.

It follows from exact replication and current traded-asset prices.

In [5]:
bond_cost_contribution = phi_B * B0
stock_cost_contribution = phi_S * S0

C0 = M0 @ phi
manual_C0 = 20.0 / 11.0

print("Bond cost contribution:", bond_cost_contribution)
print("Stock cost contribution:", stock_cost_contribution)

print()
print("Replication cost C0:", C0)
print("Hand-derived exact value 20/11:", manual_C0)

print()
print(
    "Numerical price matches hand derivation:",
    np.isclose(C0, manual_C0),
)

Bond cost contribution: -1.5151515151515147
Stock cost contribution: 3.333333333333333

Replication cost C0: 1.8181818181818183
Hand-derived exact value 20/11: 1.8181818181818181

Numerical price matches hand derivation: True


### Interpretation of the Arbitrage-Free Replication Price

The numerical calculation gives

$$
C_0
=
1.8181818181818183,
$$

while the hand-derived exact value is

$$
\frac{20}{11}
=
1.8181818181818181.
$$

Python confirms that these values agree numerically:

`Numerical price matches hand derivation: True`.

The tiny decimal difference is a floating-point representation effect.

Therefore,

$$
\boxed{
C_0
=
\frac{20}{11}
\approx
1.81818.
}
$$

---

#### 1. Decomposing the Initial Portfolio Cost

The replicating portfolio is

$$
\phi
=
\begin{pmatrix}
-\frac5{33}\\
\frac13
\end{pmatrix}.
$$

The bond position contributes

$$
\phi_B B_0
=
-\frac5{33}(10)
=
-\frac{50}{33}
\approx
-1.51515
$$

to the current portfolio cost.

The stock position contributes

$$
\phi_S S_0
=
\frac13(10)
=
\frac{10}{3}
\approx
3.33333.
$$

Adding the two components gives

$$
C_0
=
-\frac{50}{33}
+
\frac{10}{3}
=
\frac{20}{11}.
$$

Thus,

$$
\boxed{
-1.51515+3.33333
\approx
1.81818.
}
$$

---

#### 2. Why One Cost Contribution Is Negative

The negative bond-cost contribution follows from

$$
\phi_B<0.
$$

The strategy is short the bond, so the bond position generates cash at the
initial date rather than requiring a positive cash outlay.

The positive stock position requires financing.

Economically, the strategy uses the short bond position together with the long
stock position to construct the claim payoff.

The negative number should therefore not be interpreted as an error in the
pricing calculation.

It reflects the financing structure of the replicating portfolio.

---

#### 3. Replication Is What Links Hedging and Pricing

The critical result from Part 4 is

$$
M\phi=C_1.
$$

Therefore, the traded-asset portfolio and the call produce the same future
cash flow in every state:

$$
\boxed{
\text{replicating portfolio payoff}
=
\text{claim payoff}.
}
$$

Once that equality has been established, the no-arbitrage principle requires
their current values to agree as well.

Thus,

$$
\boxed{
\text{claim price}
=
\text{cost of replicating portfolio}.
}
$$

This is why the calculation

$$
C_0=\phi^\top M_0
$$

is a pricing result rather than merely an accounting calculation.

---

#### 4. The Too-Expensive Case

Suppose the claim were quoted at a price

$$
\widetilde C_0>C_0.
$$

The claim would then be more expensive than a traded portfolio with exactly
the same future cash flows.

A trader could

- sell the overpriced claim;
- buy its cheaper replicating portfolio.

The initial net cash inflow would be

$$
\widetilde C_0-C_0>0.
$$

At maturity,

$$
M\phi-C_1=0
$$

in every state.

Thus there would be a positive initial cash flow with no future net liability.

That is an arbitrage.

---

#### 5. The Too-Cheap Case

If instead

$$
\widetilde C_0<C_0,
$$

the claim would be cheaper than the portfolio with identical future cash flows.

A trader could

- buy the underpriced claim;
- short the replicating portfolio.

The initial net cash inflow would be

$$
C_0-\widetilde C_0>0.
$$

At maturity, the claim payoff offsets the short portfolio payoff exactly.

Again,

$$
C_1-M\phi=0
$$

in every state.

That would also be an arbitrage.

---

#### 6. Unique Replication Produces a Unique Price

Part 2 established that

$$
\det(M)\neq0
$$

and

$$
\operatorname{rank}(M)=2.
$$

Therefore, the replicating portfolio is unique.

Because the portfolio is unique, its current cost is also uniquely determined
by the traded asset prices.

Hence the complete two-state market produces the unique replication price

$$
\boxed{
C_0=\frac{20}{11}.
}
$$

This contrasts with an incomplete market, where an unattainable claim may not
have a unique price implied by replication alone.

---

#### 7. No Physical Probability Was Needed

The price

$$
C_0=\frac{20}{11}
$$

was derived without specifying physical probabilities for the Up and Down
states.

The reason is that replication matches the claim in every state:

$$
M\phi=C_1.
$$

Because the equality is state by state, it does not depend on which state is
more likely under a physical probability measure.

The pricing argument here is therefore

$$
\boxed{
\text{exact replication}
+
\text{no arbitrage}
\Longrightarrow
\text{claim price}.
}
$$

It is not

$$
\text{physical expected payoff}
\Longrightarrow
\text{claim price}.
$$

---

#### 8. Python Representation

The expression

`C0 = M0 @ phi`

computes the dot product

$$
M_0^\top\phi.
$$

Since

$$
M_0=
\begin{pmatrix}
10\\
10
\end{pmatrix},
$$

this is exactly

$$
10\phi_B+10\phi_S.
$$

The separate variables

`bond_cost_contribution`

and

`stock_cost_contribution`

make the financing structure visible instead of hiding the calculation inside
one matrix operation.

Finally,

`np.isclose(C0, manual_C0)`

checks that the numerical implementation agrees with the exact hand-derived
price.

---

#### 9. Central Financial Result

The complete logic developed so far is

$$
\boxed{
\text{invertible payoff matrix}
\rightarrow
\text{unique replicating portfolio}
\rightarrow
\text{identical state payoffs}
\rightarrow
\text{unique no-arbitrage price}.
}
$$

For this claim,

$$
\boxed{
\phi
=
\left(
-\frac5{33},
\frac13
\right)
}
$$

and

$$
\boxed{
C_0
=
\frac{20}{11}
\approx
1.81818.
}
$$

The next step is to package this replication-and-pricing procedure into a
reusable function that can price other claims in the same market while
performing numerical checks automatically.

## Part 6 — Building a Reusable Replication-Pricing Function

The previous sections solved and priced one particular call option.

The next objective is to turn the same financial logic into a reusable function
that can be applied to other contingent claims in the same market.

The function accepts three mathematical objects:

$$
M,
\qquad
M_0,
\qquad
C_1.
$$

Here,

$$
M
$$

is the future traded-asset payoff matrix,

$$
M_0
$$

is the vector of current traded-asset prices, and

$$
C_1
$$

is the target contingent-claim payoff vector.

The function must perform the following sequence:

$$
\boxed{
M\phi=C_1
}
$$

to determine the replicating portfolio,

then verify

$$
\boxed{
\|M\phi-C_1\|_2
\approx0,
}
$$

and finally calculate

$$
\boxed{
C_0=M_0^\top\phi.
}
$$

Thus, the computational workflow mirrors the financial argument:

$$
\boxed{
\text{claim payoff}
\rightarrow
\text{replicating holdings}
\rightarrow
\text{replication check}
\rightarrow
\text{current hedge cost}.
}
$$

---

### 1. Input Dimensions

For a market with the same number of traded assets as states, the payoff matrix
must be square.

In the present two-state model,

$$
M\in\mathbb{R}^{2\times2},
$$

while

$$
M_0\in\mathbb{R}^2
$$

and

$$
C_1\in\mathbb{R}^2.
$$

The reusable implementation checks these dimensions explicitly so that malformed
inputs fail with an interpretable error rather than producing an unclear
linear-algebra failure later.

---

### 2. Solving the Replication Problem

The function uses

`np.linalg.solve(M, C1)`

to solve

$$
M\phi=C_1.
$$

When $M$ is square and nonsingular, the solution is unique.

If the matrix is singular, `np.linalg.solve` cannot determine a unique
replicating portfolio.

The function therefore catches that numerical failure and reports it as a
financially interpretable error.

A dedicated singular-market experiment will be performed in the next section.

---

### 3. Residual Verification

A numerical solver returning a vector of holdings is not enough by itself.

The function reconstructs the portfolio payoff

$$
M\phi
$$

and calculates the residual

$$
e=M\phi-C_1.
$$

Its Euclidean norm is

$$
\|e\|_2.
$$

For exact replication, this should be zero in exact mathematics.

Because floating-point calculations can produce tiny numerical residues, the
implementation accepts replication when

$$
\|e\|_2\leq10^{-10}.
$$

The tolerance

$$
10^{-10}
$$

is an implementation convention used to distinguish machine-level rounding
error from a materially failed replication.

---

### 4. Pricing After Verification

Only after the replication check passes does the function calculate the
time-$0$ price:

$$
C_0=M_0^\top\phi.
$$

This ordering reflects the financial logic.

The portfolio cost is interpreted as the claim's no-arbitrage price because
the portfolio has first been verified to reproduce the claim's future payoff.

---

### 5. Function Output

The function returns

$$
(\phi,C_0).
$$

Thus, its output contains both

- the hedge: the traded-asset holdings $\phi$;
- the valuation: the current replication cost $C_0$.

The original call example will be passed through the function first so that its
output can be compared with the hand-derived and previously verified results.

In [6]:
def price_by_replication(M, M0, C1):
    """
    Price a claim by exact replication in a square payoff system.

    Parameters
    ----------
    M : np.ndarray
        Future traded-asset payoff matrix.
        Rows are states; columns are traded assets.
    M0 : np.ndarray
        Current traded-asset price vector.
    C1 : np.ndarray
        Target future claim payoff vector.

    Returns
    -------
    phi : np.ndarray
        Replicating portfolio holdings.
    C0 : float
        Current cost of the replicating portfolio.

    Raises
    ------
    ValueError
        If dimensions are inconsistent, the payoff matrix is singular,
        or the computed replication residual exceeds the tolerance.
    """
    M = np.asarray(M, dtype=float)
    M0 = np.asarray(M0, dtype=float)
    C1 = np.asarray(C1, dtype=float)

    if M.ndim != 2 or M.shape[0] != M.shape[1]:
        raise ValueError(
            "M must be a square two-dimensional payoff matrix."
        )

    n_states, n_assets = M.shape

    if M0.shape != (n_assets,):
        raise ValueError(
            "M0 must contain one current price for each traded asset."
        )

    if C1.shape != (n_states,):
        raise ValueError(
            "C1 must contain one payoff for each future state."
        )

    try:
        phi = np.linalg.solve(M, C1)
    except np.linalg.LinAlgError as exc:
        raise ValueError(
            "Payoff matrix M is singular; "
            "a unique replication portfolio cannot be determined."
        ) from exc

    replicated_payoff = M @ phi
    residual = replicated_payoff - C1
    residual_norm = np.linalg.norm(residual)

    tolerance = 1e-10

    if residual_norm > tolerance:
        raise ValueError(
            "Replication residual exceeds the numerical tolerance."
        )

    C0 = M0 @ phi

    return phi, C0


phi_function, C0_function = price_by_replication(M, M0, C1)

function_payoff = M @ phi_function
function_residual = function_payoff - C1
function_residual_norm = np.linalg.norm(function_residual)

print("Function replicating portfolio:", phi_function)
print("Function claim price:", C0_function)

print()
print("Function replicated payoff:", function_payoff)
print("Function residual:", function_residual)
print("Function residual norm:", function_residual_norm)

print()
print(
    "Function holdings match earlier solution:",
    np.allclose(phi_function, phi),
)
print(
    "Function price matches earlier price:",
    np.isclose(C0_function, C0),
)

Function replicating portfolio: [-0.15151515  0.33333333]
Function claim price: 1.8181818181818183

Function replicated payoff: [5.00000000e+00 4.16333634e-16]
Function residual: [0.00000000e+00 4.16333634e-16]
Function residual norm: 4.163336342344337e-16

Function holdings match earlier solution: True
Function price matches earlier price: True


### Interpretation of the Reusable Replication-Pricing Function

The reusable function returns the portfolio

$$
\phi
\approx
\begin{pmatrix}
-0.15151515\\
0.33333333
\end{pmatrix}
$$

and the claim price

$$
C_0
\approx
1.81818182.
$$

These values match the results derived independently in Parts 3–5.

Python confirms

`Function holdings match earlier solution: True`

and

`Function price matches earlier price: True`.

Therefore,

$$
\boxed{
\phi
=
\begin{pmatrix}
-\frac5{33}\\
\frac13
\end{pmatrix}
}
$$

and

$$
\boxed{
C_0
=
\frac{20}{11}
\approx
1.81818.
}
$$

---

#### 1. The Function Encodes the Full Replication Argument

The function

`price_by_replication(M, M0, C1)`

does more than evaluate a single formula.

It implements the full logical sequence

$$
\boxed{
M\phi=C_1
\rightarrow
\text{verify replication}
\rightarrow
C_0=M_0^\top\phi.
}
$$

Thus, the function mirrors the financial reasoning developed earlier in the
notebook.

The claim is priced only after a portfolio has been found and verified to
reproduce its payoff.

---

#### 2. Input Roles

The function accepts three distinct financial objects.

The payoff matrix

$$
M
$$

contains the future payoffs of the traded assets.

The current price vector

$$
M_0
$$

contains the current market prices of those traded assets.

The claim vector

$$
C_1
$$

contains the future payoff that must be replicated.

These inputs therefore play different roles:

$$
\boxed{
M,\;C_1
\text{ determine the hedge}
}
$$

while

$$
\boxed{
M_0
\text{ determines the current hedge cost}.
}
$$

Keeping these roles separate prevents current prices and future payoffs from
being mixed together.

---

#### 3. Dimension Checks

Before attempting to solve the replication system, the function verifies that
the inputs are structurally compatible.

For the present market,

$$
M\in\mathbb{R}^{2\times2},
$$

$$
M_0\in\mathbb{R}^2,
$$

and

$$
C_1\in\mathbb{R}^2.
$$

The dimension checks confirm that

- $M$ is square;
- $M_0$ contains one current price per traded asset;
- $C_1$ contains one payoff per future state.

These checks make the function fail early and clearly when the financial
objects do not have compatible dimensions.

---

#### 4. Converting Inputs With `np.asarray`

Inside the function,

`np.asarray(..., dtype=float)`

converts the supplied inputs into NumPy arrays with floating-point numerical
storage.

This gives the linear-algebra operations a consistent numerical
representation.

The conversion does not change the financial meaning of the inputs.

It standardizes their computational form.

---

#### 5. Solving for the Replicating Holdings

The function uses

`np.linalg.solve(M, C1)`

to solve

$$
M\phi=C_1.
$$

Because the current matrix is nonsingular, the solver returns the unique
replicating portfolio.

The result is

$$
\phi
=
\begin{pmatrix}
-\frac5{33}\\
\frac13
\end{pmatrix}.
$$

This agrees with the hand derivation.

Thus, the function is not introducing a different pricing method.

It is automating the same replication problem already solved analytically.

---

#### 6. Singular-Matrix Handling

The call to `np.linalg.solve` is placed inside a `try`/`except` structure.

If the payoff matrix is singular, NumPy raises a linear-algebra error because a
unique solution cannot be determined.

The function converts that low-level numerical failure into the more
financially meaningful message that a unique replication portfolio cannot be
determined.

This distinction matters because a singular payoff matrix is not merely a
software problem.

It reflects a structural problem in the traded payoff system.

The next section examines that case explicitly.

---

#### 7. Residual Verification

After solving for $\phi$, the function reconstructs the payoff

$$
M\phi
$$

and compares it with the target claim

$$
C_1.
$$

The residual is

$$
e=M\phi-C_1.
$$

Python returns approximately

$$
e=
\begin{pmatrix}
0\\
4.16\times10^{-16}
\end{pmatrix}.
$$

Its norm is

$$
\|e\|_2
\approx
4.16\times10^{-16}.
$$

This is machine-level floating-point residue.

Therefore,

$$
\boxed{
\|e\|_2\approx0.
}
$$

The function accepts the replication because the residual norm is far below the
chosen tolerance

$$
10^{-10}.
$$

---

#### 8. Why the Tolerance Is Needed

In exact mathematics,

$$
M\phi=C_1
$$

would produce a zero residual.

In floating-point computation, operations involving fractions such as

$$
\frac13
$$

cannot generally be represented with infinite precision.

Therefore, tiny residuals can appear even when the underlying mathematical
replication is exact.

The tolerance allows the implementation to distinguish

$$
\text{numerical rounding noise}
$$

from

$$
\text{a materially failed replication}.
$$

The value

$$
10^{-10}
$$

is an implementation convention in this notebook, not a universal financial
constant.

---

#### 9. Pricing Occurs Only After Replication Is Verified

Once the residual check passes, the function computes

$$
C_0=M_0^\top\phi.
$$

For this market,

$$
M_0=
\begin{pmatrix}
10\\
10
\end{pmatrix},
$$

so

$$
C_0
=
10\left(-\frac5{33}\right)
+
10\left(\frac13\right)
=
\frac{20}{11}.
$$

Thus,

$$
\boxed{
C_0
=
\frac{20}{11}
\approx1.81818.
}
$$

The ordering of the calculations is important:

$$
\boxed{
\text{solve}
\rightarrow
\text{verify}
\rightarrow
\text{price}.
}
$$

This makes the financial logic explicit rather than treating the function as a
black box.

---

#### 10. Meaning of the Returned Values

The function returns

`phi, C0`.

The first returned object,

$$
\phi,
$$

is the replicating portfolio.

The second,

$$
C_0,
$$

is the current cost of that portfolio.

Thus, the function produces both

$$
\boxed{
\text{hedging information}
}
$$

and

$$
\boxed{
\text{pricing information}.
}
$$

These are related because the price is derived from the verified hedge.

---

#### 11. Why Reusability Matters

The function is no longer tied to the specific call payoff

$$
C_1=(5,0).
$$

As long as the same complete market matrix $M$ and current price vector $M_0$
are used, another two-state claim can be supplied as a new $C_1$.

The function will then

1. solve for its replicating portfolio;
2. verify the payoff match;
3. calculate its replication price.

This makes the notebook capable of testing many claims without rewriting the
financial logic each time.

That reusability is required for the later linearity and stress-test
experiments.

---

#### 12. Central Result

The reusable function preserves the core Chapter 2 principle:

$$
\boxed{
\text{exact replication}
+
\text{current traded-asset prices}
\Longrightarrow
\text{no-arbitrage claim price}.
}
$$

For the original call example, the function independently reproduces

$$
\boxed{
\phi=
\left(
-\frac5{33},
\frac13
\right)
}
$$

and

$$
\boxed{
C_0=\frac{20}{11}.
}
$$

The next section deliberately breaks the market payoff structure by making the
matrix singular.

This will show why the existence of a numerical function does not guarantee
that a unique replication problem is economically well defined.

## Part 7 — Singular Payoff Matrices and Controlled Failure

The reusable pricing function assumes that the traded-asset payoff matrix
provides enough independent payoff directions to determine a unique
replicating portfolio.

The original market satisfies this condition because

$$
\operatorname{rank}(M)=2
$$

in the two-dimensional payoff space

$$
\mathbb{R}^2.
$$

The capstone now deliberately constructs a singular payoff matrix to examine
what fails when the traded assets become redundant.

---

### 1. A Deliberately Singular Market

Consider

$$
M_{\mathrm{sing}}
=
\begin{pmatrix}
11&22\\
11&22
\end{pmatrix}.
$$

The first traded asset has payoff vector

$$
\begin{pmatrix}
11\\
11
\end{pmatrix},
$$

while the second has payoff vector

$$
\begin{pmatrix}
22\\
22
\end{pmatrix}.
$$

But

$$
\begin{pmatrix}
22\\
22
\end{pmatrix}
=
2
\begin{pmatrix}
11\\
11
\end{pmatrix}.
$$

Therefore the second payoff vector does not add a new payoff direction.

The two columns are linearly dependent.

---

### 2. Determinant and Rank

The determinant is

$$
\det(M_{\mathrm{sing}})
=
11(22)-22(11)
=
0.
$$

Hence,

$$
\boxed{
\det(M_{\mathrm{sing}})=0.
}
$$

The matrix therefore has no inverse.

Its rank is

$$
\boxed{
\operatorname{rank}(M_{\mathrm{sing}})=1.
}
$$

Although two assets are listed, they span only a one-dimensional payoff
subspace.

Financially, the second asset is redundant with the first in terms of future
state payoffs.

---

### 3. The Attainable Payoff Space

Any portfolio

$$
\phi=
\begin{pmatrix}
\phi_1\\
\phi_2
\end{pmatrix}
$$

produces

$$
M_{\mathrm{sing}}\phi
=
\begin{pmatrix}
11\phi_1+22\phi_2\\
11\phi_1+22\phi_2
\end{pmatrix}.
$$

Therefore every attainable payoff has the form

$$
\boxed{
(x,x)
}
$$

for some scalar $x$.

The market can generate only payoffs that are identical in the Up and Down
states.

It cannot generate arbitrary vectors in

$$
\mathbb{R}^2.
$$

---

### 4. Why the Selected Claim Cannot Be Replicated

The target call claim remains

$$
C_1=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
$$

This payoff is not of the form

$$
(x,x).
$$

Therefore,

$$
\boxed{
C_1\notin\operatorname{col}(M_{\mathrm{sing}}).
}
$$

The replication system

$$
M_{\mathrm{sing}}\phi=C_1
$$

has no exact solution for this particular claim.

This can also be diagnosed by comparing ranks.

The market matrix has

$$
\operatorname{rank}(M_{\mathrm{sing}})=1,
$$

while the augmented matrix

$$
[M_{\mathrm{sing}}\mid C_1]
$$

has rank $2$.

Thus,

$$
\boxed{
\operatorname{rank}(M_{\mathrm{sing}})
<
\operatorname{rank}([M_{\mathrm{sing}}\mid C_1]),
}
$$

which means the linear system is inconsistent.

---

### 5. Singularity Does Not Always Mean “No Solution”

A singular matrix does not imply that every claim is impossible to replicate.

For example, a claim of the form

$$
(x,x)
$$

does lie in the attainable payoff space.

Such a claim could have multiple portfolios producing the same payoff because
the two traded payoff columns are redundant.

Therefore, singularity means that a **unique** replication portfolio cannot in
general be determined.

Depending on the claim, the system may have

- no solution; or
- infinitely many solutions.

For the selected claim

$$
C_1=(5,0),
$$

the result is specifically **no exact solution**.

---

### 6. Controlled Failure in the Pricing Function

The function uses

`np.linalg.solve(...)`

only when the square payoff system can support a unique solution.

For a singular matrix, NumPy raises a linear-algebra error.

The reusable function catches that low-level numerical error and converts it
into a `ValueError` with a financially meaningful explanation.

The test below catches that expected `ValueError` so that the notebook can
demonstrate the failure without interrupting a full top-to-bottom execution.

A successful test should therefore show

$$
\boxed{
\text{singular market rejected}=\mathrm{True}.
}
$$

In [7]:
M_singular = np.array([
    [11.0, 22.0],
    [11.0, 22.0],
])

det_singular = np.linalg.det(M_singular)
rank_singular = np.linalg.matrix_rank(M_singular)

augmented_singular = np.column_stack((M_singular, C1))
rank_augmented_singular = np.linalg.matrix_rank(augmented_singular)

columns_redundant = np.allclose(
    M_singular[:, 1],
    2.0 * M_singular[:, 0],
)

print("Singular payoff matrix:")
print(M_singular)

print()
print("Determinant:", det_singular)
print("Rank of singular matrix:", rank_singular)
print("Rank of augmented matrix:", rank_augmented_singular)
print("Second payoff column is twice the first:", columns_redundant)

print()
print(
    "Selected claim exactly attainable:",
    rank_singular == rank_augmented_singular,
)

singular_market_rejected = False

try:
    price_by_replication(M_singular, M0, C1)

except ValueError as exc:
    singular_market_rejected = True
    print()
    print("Controlled pricing failure:")
    print(exc)

print()
print("Singular market rejected:", singular_market_rejected)

Singular payoff matrix:
[[11. 22.]
 [11. 22.]]

Determinant: 0.0
Rank of singular matrix: 1
Rank of augmented matrix: 2
Second payoff column is twice the first: True

Selected claim exactly attainable: False

Controlled pricing failure:
Payoff matrix M is singular; a unique replication portfolio cannot be determined.

Singular market rejected: True


### Interpretation of Singularity and Controlled Failure

The deliberately constructed payoff matrix is

$$
M_{\mathrm{sing}}
=
\begin{pmatrix}
11&22\\
11&22
\end{pmatrix}.
$$

Python confirms

$$
\det(M_{\mathrm{sing}})=0
$$

and

$$
\operatorname{rank}(M_{\mathrm{sing}})=1.
$$

The second payoff column is exactly twice the first, so the two traded assets
do not provide two independent payoff directions.

---

#### 1. Redundant Traded Payoffs

The two columns are

$$
\begin{pmatrix}
11\\
11
\end{pmatrix}
$$

and

$$
\begin{pmatrix}
22\\
22
\end{pmatrix}.
$$

Because

$$
\begin{pmatrix}
22\\
22
\end{pmatrix}
=
2
\begin{pmatrix}
11\\
11
\end{pmatrix},
$$

the second traded payoff adds no new direction to the attainable payoff space.

Therefore,

$$
\boxed{
\operatorname{rank}(M_{\mathrm{sing}})=1.
}
$$

Although the model lists two traded assets, their future payoffs span only a
one-dimensional subspace of

$$
\mathbb{R}^2.
$$

---

#### 2. The Attainable Payoff Space Collapses to a Line

For arbitrary holdings

$$
\phi=
\begin{pmatrix}
\phi_1\\
\phi_2
\end{pmatrix},
$$

the resulting future payoff is

$$
M_{\mathrm{sing}}\phi
=
\begin{pmatrix}
11\phi_1+22\phi_2\\
11\phi_1+22\phi_2
\end{pmatrix}.
$$

Both state payoffs must therefore be equal.

Every attainable payoff has the form

$$
\boxed{
(x,x).
}
$$

Geometrically, the attainable set is a one-dimensional line inside the
two-dimensional contingent-claim space.

Thus,

$$
\boxed{
\operatorname{col}(M_{\mathrm{sing}})
\subsetneq
\mathbb{R}^2.
}
$$

The singular market is therefore incomplete under the unrestricted
portfolio-holding framework used in this notebook.

---

#### 3. Why the Selected Claim Is Unattainable

The target claim is

$$
C_1=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
$$

Its two state payoffs are different.

Therefore it cannot have the form

$$
(x,x).
$$

Hence,

$$
\boxed{
C_1\notin\operatorname{col}(M_{\mathrm{sing}}).
}
$$

The selected claim cannot be replicated exactly using these redundant traded
payoffs.

---

#### 4. Rank Test for Inconsistency

Python reports

$$
\operatorname{rank}(M_{\mathrm{sing}})=1
$$

while

$$
\operatorname{rank}
\left(
[M_{\mathrm{sing}}\mid C_1]
\right)
=
2.
$$

Adding the claim payoff therefore introduces a new independent payoff
direction.

Because

$$
\operatorname{rank}(M_{\mathrm{sing}})
<
\operatorname{rank}
\left(
[M_{\mathrm{sing}}\mid C_1]
\right),
$$

the system

$$
M_{\mathrm{sing}}\phi=C_1
$$

is inconsistent.

Thus,

$$
\boxed{
\text{no exact replicating portfolio exists for }C_1=(5,0).
}
$$

---

#### 5. Why Singularity and Unattainability Are Related but Distinct

The singularity of the market matrix means that the traded payoff vectors are
linearly dependent.

That alone does not mean that every claim has no solution.

A claim such as

$$
C_1=(x,x)
$$

does lie in the one-dimensional attainable payoff space.

For such a claim, the redundant assets could generate multiple portfolios with
the same future payoff.

Therefore a singular system can produce either

- no solution for a claim outside the attainable payoff space; or
- multiple solutions for a claim inside that space.

For the particular claim studied here,

$$
C_1=(5,0),
$$

the rank test establishes the first case:

$$
\boxed{
\text{no solution}.
}
$$

---

#### 6. Numerical Failure Reflects Financial Structure

When the function calls

`np.linalg.solve(M_singular, M0, C1)`

conceptually, the problem is not simply that NumPy dislikes the matrix.

The numerical solver fails because the underlying linear system does not have
the unique solution required by the function's replication-pricing framework.

The singularity therefore has a financial interpretation:

$$
\boxed{
\text{the traded assets do not provide enough independent payoff directions}.
}
$$

The numerical error is a computational manifestation of that structural
financial fact.

---

#### 7. Controlled Error Handling

Rather than allowing the low-level NumPy error to terminate the notebook, the
pricing function converts it into the message:

`Payoff matrix M is singular; a unique replication portfolio cannot be determined.`

The surrounding `try`/`except` test then catches that expected error.

This is deliberate behavior.

A singular market is not treated as though a valid replication price had been
computed.

Instead,

$$
\boxed{
\text{invalid unique-replication problem}
\rightarrow
\text{explicit controlled rejection}.
}
$$

This allows the notebook to communicate the failure clearly while still
remaining reproducible under a complete top-to-bottom execution.

---

#### 8. Why the Function Rejects Singular Matrices

The purpose of `price_by_replication(...)` is specifically to implement
**unique exact replication pricing in a square payoff system**.

A singular matrix falls outside that function's intended domain because the
mapping from claim payoff to a unique portfolio is no longer well defined.

Therefore the function does not attempt to choose among multiple possible
solutions or replace exact replication with an approximation method.

Those would be different financial and numerical problems requiring additional
rules.

The function instead stops and reports that its assumptions have failed.

---

#### 9. Contrast With the Original Market

The original payoff matrix satisfies

$$
\operatorname{rank}(M)=2,
$$

so

$$
\operatorname{col}(M)=\mathbb{R}^2.
$$

The singular matrix satisfies

$$
\operatorname{rank}(M_{\mathrm{sing}})=1,
$$

so

$$
\operatorname{col}(M_{\mathrm{sing}})
\subsetneq
\mathbb{R}^2.
$$

Thus the contrast is

$$
\boxed{
\begin{array}{c|c|c}
\text{Market} & \text{Rank} & \text{Replication structure}\\
\hline
M & 2 & \text{every two-state claim attainable}\\
M_{\mathrm{sing}} & 1 & \text{only a one-dimensional set attainable}
\end{array}
}
$$

This makes the relationship between linear independence and market
completeness concrete.

---

#### 10. Central Result

The singularity experiment demonstrates that the success of replication
pricing depends on the structure of the traded payoff system.

The chain is

$$
\boxed{
\text{linearly dependent traded payoffs}
\rightarrow
\text{rank deficiency}
\rightarrow
\text{restricted attainable payoff space}
\rightarrow
\text{failure of unique replication}.
}
$$

For the selected claim,

$$
C_1=(5,0),
$$

the failure is stronger: the claim lies outside the attainable payoff space, so
no exact hedge exists.

The next section returns to the original complete market and examines another
fundamental property of replication pricing: linearity.

## Part 8 — Linearity of Replication Pricing

The original market is complete, so every two-state contingent claim has a
unique replicating portfolio.

This structure implies an important property of the pricing rule:
**linearity**.

To see why, consider two claims

$$
A_1
$$

and

$$
B_1.
$$

Suppose their replicating portfolios are

$$
\phi^A
$$

and

$$
\phi^B,
$$

so that

$$
M\phi^A=A_1
$$

and

$$
M\phi^B=B_1.
$$

---

### 1. Replication of the Sum

Because matrix multiplication is linear,

$$
M(\phi^A+\phi^B)
=
M\phi^A+M\phi^B.
$$

Using the replication equations,

$$
M(\phi^A+\phi^B)
=
A_1+B_1.
$$

Therefore,

$$
\phi^A+\phi^B
$$

replicates the combined claim

$$
A_1+B_1.
$$

Because the market payoff matrix is invertible, the replication portfolio is
unique.

Hence,

$$
\boxed{
\phi^{A+B}
=
\phi^A+\phi^B.
}
$$

---

### 2. Pricing the Sum

Replication pricing assigns a claim the current cost of its replicating
portfolio.

Therefore,

$$
p(A_1)
=
M_0^\top\phi^A
$$

and

$$
p(B_1)
=
M_0^\top\phi^B.
$$

For the combined claim,

$$
p(A_1+B_1)
=
M_0^\top\phi^{A+B}.
$$

Using

$$
\phi^{A+B}
=
\phi^A+\phi^B,
$$

we obtain

$$
p(A_1+B_1)
=
M_0^\top(\phi^A+\phi^B).
$$

By distributivity of the dot product,

$$
p(A_1+B_1)
=
M_0^\top\phi^A
+
M_0^\top\phi^B.
$$

Therefore,

$$
\boxed{
p(A_1+B_1)
=
p(A_1)+p(B_1).
}
$$

This is the additivity property that will be tested numerically.

---

### 3. A Concrete Pair of Claims

For the numerical example, choose

$$
A_1=
\begin{pmatrix}
5\\
0
\end{pmatrix},
$$

which is the original call claim, and

$$
B_1=
\begin{pmatrix}
0\\
3
\end{pmatrix}.
$$

Their combined payoff is

$$
A_1+B_1
=
\begin{pmatrix}
5\\
3
\end{pmatrix}.
$$

The reusable pricing function will independently compute

- the replication portfolio for $A_1$;
- the replication portfolio for $B_1$;
- the replication portfolio for $A_1+B_1$;
- the corresponding three prices.

The numerical checks will then verify both

$$
\boxed{
\phi^{A+B}
=
\phi^A+\phi^B
}
$$

and

$$
\boxed{
p(A_1+B_1)
=
p(A_1)+p(B_1).
}
$$

---

### 4. Why This Property Matters

Linearity means that the valuation rule is internally consistent with combining
and scaling cash flows.

If two claims are combined into one portfolio of claims, the value of the
combined payoff should equal the sum of their separately replicated values.

This property follows from the linear structure of

$$
M\phi=C_1
$$

and the linear cost calculation

$$
C_0=M_0^\top\phi.
$$

The next code cell verifies this result numerically before the property is
tested across many randomly generated claims.

In [8]:
claim_A = np.array([5.0, 0.0])
claim_B = np.array([0.0, 3.0])
claim_sum = claim_A + claim_B

phi_A, price_A = price_by_replication(M, M0, claim_A)
phi_B, price_B = price_by_replication(M, M0, claim_B)
phi_sum, price_sum = price_by_replication(M, M0, claim_sum)

holdings_additive = np.allclose(
    phi_sum,
    phi_A + phi_B,
)

prices_additive = np.isclose(
    price_sum,
    price_A + price_B,
)

print("Claim A:", claim_A)
print("Replicating portfolio for A:", phi_A)
print("Price of A:", price_A)

print()
print("Claim B:", claim_B)
print("Replicating portfolio for B:", phi_B)
print("Price of B:", price_B)

print()
print("Combined claim A + B:", claim_sum)
print("Replicating portfolio for A + B:", phi_sum)
print("Price of A + B:", price_sum)

print()
print("phi_A + phi_B:", phi_A + phi_B)
print("price_A + price_B:", price_A + price_B)

print()
print("Replication holdings are additive:", holdings_additive)
print("Replication prices are additive:", prices_additive)

Claim A: [5. 0.]
Replicating portfolio for A: [-0.15151515  0.33333333]
Price of A: 1.8181818181818183

Claim B: [0. 3.]
Replicating portfolio for B: [ 0.36363636 -0.2       ]
Price of B: 1.6363636363636367

Combined claim A + B: [5. 3.]
Replicating portfolio for A + B: [0.21212121 0.13333333]
Price of A + B: 3.454545454545454

phi_A + phi_B: [0.21212121 0.13333333]
price_A + price_B: 3.454545454545455

Replication holdings are additive: True
Replication prices are additive: True


### Interpretation of Linearity in Replication Pricing

The numerical experiment confirms both forms of additivity:

$$
\phi^{A+B}
=
\phi^A+\phi^B
$$

and

$$
p(A_1+B_1)
=
p(A_1)+p(B_1).
$$

Python reports

`Replication holdings are additive: True`

and

`Replication prices are additive: True`.

Thus, the computational results agree with the linear-algebra derivation.

---

#### 1. Claim A

The first claim is

$$
A_1=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
$$

Its replicating portfolio is

$$
\phi^A
\approx
\begin{pmatrix}
-0.15151515\\
0.33333333
\end{pmatrix}.
$$

In exact form,

$$
\boxed{
\phi^A
=
\begin{pmatrix}
-\frac5{33}\\
\frac13
\end{pmatrix}.
}
$$

Its replication price is

$$
\boxed{
p(A_1)
=
\frac{20}{11}
\approx1.81818.
}
$$

This is the original call claim analyzed earlier in the notebook.

---

#### 2. Claim B

The second claim is

$$
B_1=
\begin{pmatrix}
0\\
3
\end{pmatrix}.
$$

Its replication equations are

$$
11\phi_B+20\phi_S=0
$$

and

$$
11\phi_B+5\phi_S=3.
$$

Subtracting the second equation from the first gives

$$
15\phi_S=-3,
$$

so

$$
\boxed{
\phi_S=-\frac15.
}
$$

Substituting into the Down-state equation,

$$
11\phi_B
+
5\left(-\frac15\right)
=
3,
$$

which gives

$$
11\phi_B-1=3.
$$

Therefore,

$$
\boxed{
\phi_B=\frac4{11}.
}
$$

Hence,

$$
\boxed{
\phi^B=
\begin{pmatrix}
\frac4{11}\\
-\frac15
\end{pmatrix}
}
$$

or numerically,

$$
\phi^B
\approx
\begin{pmatrix}
0.36363636\\
-0.2
\end{pmatrix}.
$$

Its current price is

$$
p(B_1)
=
10\left(\frac4{11}\right)
+
10\left(-\frac15\right).
$$

Thus,

$$
p(B_1)
=
\frac{40}{11}-2
=
\frac{18}{11}.
$$

Therefore,

$$
\boxed{
p(B_1)
=
\frac{18}{11}
\approx1.63636.
}
$$

---

#### 3. Combined Claim

The combined payoff is

$$
A_1+B_1
=
\begin{pmatrix}
5\\
3
\end{pmatrix}.
$$

Python returns the replicating portfolio

$$
\phi^{A+B}
\approx
\begin{pmatrix}
0.21212121\\
0.13333333
\end{pmatrix}.
$$

Now add the two individual portfolios:

$$
\phi^A+\phi^B
=
\begin{pmatrix}
-\frac5{33}\\
\frac13
\end{pmatrix}
+
\begin{pmatrix}
\frac4{11}\\
-\frac15
\end{pmatrix}.
$$

For the bond holding,

$$
-\frac5{33}+\frac4{11}
=
-\frac5{33}+\frac{12}{33}
=
\frac7{33}.
$$

For the stock holding,

$$
\frac13-\frac15
=
\frac2{15}.
$$

Therefore,

$$
\boxed{
\phi^A+\phi^B
=
\begin{pmatrix}
\frac7{33}\\
\frac2{15}
\end{pmatrix}.
}
$$

Numerically,

$$
\begin{pmatrix}
\frac7{33}\\
\frac2{15}
\end{pmatrix}
\approx
\begin{pmatrix}
0.21212121\\
0.13333333
\end{pmatrix},
$$

which matches the portfolio returned directly for the combined claim.

Hence,

$$
\boxed{
\phi^{A+B}
=
\phi^A+\phi^B.
}
$$

---

#### 4. Additivity of Prices

The individual prices are

$$
p(A_1)=\frac{20}{11}
$$

and

$$
p(B_1)=\frac{18}{11}.
$$

Therefore,

$$
p(A_1)+p(B_1)
=
\frac{20}{11}
+
\frac{18}{11}
=
\frac{38}{11}.
$$

Thus,

$$
\boxed{
p(A_1)+p(B_1)
=
\frac{38}{11}
\approx3.45455.
}
$$

Python independently prices the combined claim as

$$
p(A_1+B_1)
\approx
3.454545454545454.
$$

Therefore,

$$
\boxed{
p(A_1+B_1)
=
p(A_1)+p(B_1).
}
$$

---

#### 5. Why the Two Printed Decimals Differ Slightly

Python prints

$$
p(A_1+B_1)
=
3.454545454545454
$$

and

$$
p(A_1)+p(B_1)
=
3.454545454545455.
$$

These decimal representations differ only in the final displayed floating-point
digit.

Mathematically, both represent

$$
\frac{38}{11}.
$$

This is why the numerical comparison uses

`np.isclose(...)`

rather than requiring exact equality of the stored floating-point values.

The result

`Replication prices are additive: True`

confirms that the difference is numerical rounding noise rather than a failure
of the pricing relationship.

---

#### 6. Why Linearity Follows From Replication

The additivity result is not an accidental feature of these particular claims.

It follows from two linear operations.

First, future portfolio payoffs are linear in holdings:

$$
M(\phi^A+\phi^B)
=
M\phi^A+M\phi^B.
$$

Second, current portfolio cost is linear in holdings:

$$
M_0^\top(\phi^A+\phi^B)
=
M_0^\top\phi^A
+
M_0^\top\phi^B.
$$

Therefore the replication-pricing map inherits the linearity of the underlying
portfolio system.

Symbolically,

$$
\boxed{
p(A_1+B_1)=p(A_1)+p(B_1).
}
$$

---

#### 7. Linearity Extends Beyond Addition

The same reasoning also implies homogeneity.

For a scalar $\alpha$,

$$
M(\alpha\phi^A)
=
\alpha M\phi^A
=
\alpha A_1.
$$

Therefore,

$$
\alpha\phi^A
$$

replicates

$$
\alpha A_1.
$$

Its price is

$$
p(\alpha A_1)
=
M_0^\top(\alpha\phi^A)
=
\alpha M_0^\top\phi^A.
$$

Hence,

$$
\boxed{
p(\alpha A_1)
=
\alpha p(A_1).
}
$$

Together, additivity and homogeneity imply the more general linearity relation

$$
\boxed{
p(\alpha A_1+\beta B_1)
=
\alpha p(A_1)+\beta p(B_1).
}
$$

The capstone's numerical experiment focuses on additivity, but it is part of
this broader linear pricing structure.

---

#### 8. Financial Interpretation

Linearity means that pricing is consistent with constructing portfolios of
claims.

If two claims are held together, the combined payoff is simply the sum of
their individual payoffs.

Because their replicating portfolios can also be combined, the cost of
replicating the combined claim must equal the sum of the individual replication
costs.

Thus,

$$
\boxed{
\text{combining cash flows}
\longleftrightarrow
\text{combining replicating portfolios}
\longleftrightarrow
\text{adding prices}.
}
$$

This is a fundamental consequence of linear portfolio formation in the
complete market.

---

#### 9. Transition to the Stress Test

The concrete example verifies linearity for one pair of claims.

The next section asks whether the same relationship continues to hold across a
larger collection of automatically generated claims.

A fixed random seed will be used to create 100 reproducible two-state claim
payoffs.

The claims will then be grouped into pairs, and the pricing function will test

$$
p(A+B)
=
p(A)+p(B)
$$

repeatedly.

This moves the notebook from a single illustrative example to a systematic
computational stress test of the same theoretical property.

## Part 9 — Reproducible 100-Claim Linearity Stress Test

Part 8 established analytically that replication pricing is linear:

$$
p(A_1+B_1)
=
p(A_1)+p(B_1).
$$

A single numerical example confirmed the result for one pair of claims.

This section performs a larger computational stress test to check whether the
reusable pricing implementation preserves that property across many different
claim payoffs.

The experiment is an implementation test of a theoretical property already
derived above. It is not being used as a numerical proof of linearity.

---

### 1. Generating 100 Claims

A two-state contingent claim is represented by a payoff vector

$$
C_1=
\begin{pmatrix}
C_1^u\\
C_1^d
\end{pmatrix}
\in\mathbb{R}^2.
$$

For this experiment, 100 such payoff vectors are generated.

Each payoff component is sampled from the interval

$$
[0,10).
$$

Thus the generated object has shape

$$
(100,2),
$$

meaning

- 100 claims;
- two state payoffs per claim.

These randomly generated vectors are generic contingent claims.

They are not required to correspond to calls, puts, or any other specific
derivative contract.

Because the original market is complete,

$$
\operatorname{col}(M)=\mathbb{R}^2,
$$

every generated two-state claim is attainable in the model.

---

### 2. Fixed Seed and Reproducibility

The experiment uses the fixed random seed

$$
\boxed{42}.
$$

Using a fixed seed means that the same pseudo-random claim sample is generated
whenever the notebook is executed under the same instructions.

This allows the stress test to be reproduced exactly.

The random sample is used only as a controlled collection of test inputs.

It should not be interpreted as an empirical distribution of real financial
claims.

---

### 3. Pairing Convention

The 100 generated claims are grouped sequentially into 50 non-overlapping
pairs:

$$
(C_1^{(0)},C_1^{(1)}),
$$

$$
(C_1^{(2)},C_1^{(3)}),
$$

and so on, ending with

$$
(C_1^{(98)},C_1^{(99)}).
$$

Thus,

$$
100\text{ claims}
\longrightarrow
50\text{ claim pairs}.
$$

For every pair $(A_1,B_1)$, the function independently computes

$$
p(A_1),
$$

$$
p(B_1),
$$

and

$$
p(A_1+B_1).
$$

The test then evaluates

$$
\boxed{
p(A_1+B_1)
\stackrel{?}{=}
p(A_1)+p(B_1).
}
$$

---

### 4. Numerical Discrepancy

Because the calculations use floating-point arithmetic, the two sides may
differ by extremely small machine-level rounding amounts.

For each pair, define

$$
\delta
=
p(A_1+B_1)
-
\left[p(A_1)+p(B_1)\right].
$$

The experiment records both

$$
\delta
$$

and

$$
|\delta|.
$$

The comparison uses `np.isclose(...)` rather than requiring exact equality of
floating-point representations.

The maximum absolute discrepancy across all 50 pairs will also be reported.

---

### 5. Success Criterion

The implementation passes the stress test if all 50 pairs satisfy the
linearity comparison within numerical tolerance:

$$
\boxed{
\text{all 50 pair tests}=\mathrm{True}.
}
$$

If the theoretical pricing rule is linear but the implementation fails this
experiment materially, that would indicate a problem in the code rather than
a failure of the financial theory.

In [9]:
stress_test_seed = 42

np.random.seed(stress_test_seed)

random_claims = np.random.uniform(
    low=0.0,
    high=10.0,
    size=(100, 2),
)

linearity_records = []

for pair_number, i in enumerate(range(0, 100, 2), start=1):
    claim_A_test = random_claims[i]
    claim_B_test = random_claims[i + 1]
    combined_claim_test = claim_A_test + claim_B_test

    _, price_A_test = price_by_replication(
        M, M0, claim_A_test
    )

    _, price_B_test = price_by_replication(
        M, M0, claim_B_test
    )

    _, price_sum_test = price_by_replication(
        M, M0, combined_claim_test
    )

    separate_price_sum = price_A_test + price_B_test

    discrepancy = (
        price_sum_test - separate_price_sum
    )

    absolute_discrepancy = abs(discrepancy)

    linearity_holds = np.isclose(
        price_sum_test,
        separate_price_sum,
    )

    linearity_records.append(
        {
            "pair": pair_number,
            "price_A": price_A_test,
            "price_B": price_B_test,
            "price_A_plus_B": price_sum_test,
            "separate_price_sum": separate_price_sum,
            "discrepancy": discrepancy,
            "absolute_discrepancy": absolute_discrepancy,
            "linearity_holds": linearity_holds,
        }
    )

all_pairs_linear = all(
    record["linearity_holds"]
    for record in linearity_records
)

max_absolute_discrepancy = max(
    record["absolute_discrepancy"]
    for record in linearity_records
)

print("Random seed:", stress_test_seed)
print("Random claims shape:", random_claims.shape)

print()
print("First five generated claims:")
print(random_claims[:5])

print()
print("Number of claim pairs tested:", len(linearity_records))
print("All pairs satisfy linearity:", all_pairs_linear)
print(
    "Maximum absolute pricing discrepancy:",
    max_absolute_discrepancy,
)

print()
print("First five pairwise tests:")

for record in linearity_records[:5]:
    print()
    print("Pair:", record["pair"])
    print("Price A:", record["price_A"])
    print("Price B:", record["price_B"])
    print("Price A + B:", record["price_A_plus_B"])
    print(
        "Price A + Price B:",
        record["separate_price_sum"],
    )
    print("Discrepancy:", record["discrepancy"])
    print(
        "Linearity holds:",
        record["linearity_holds"],
    )

Random seed: 42
Random claims shape: (100, 2)

First five generated claims:
[[3.74540119 9.50714306]
 [7.31993942 5.98658484]
 [1.5601864  1.5599452 ]
 [0.58083612 8.66176146]
 [6.01115012 7.08072578]]

Number of claim pairs tested: 50
All pairs satisfy linearity: True
Maximum absolute pricing discrepancy: 3.552713678800501e-15

First five pairwise tests:

Pair: 1
Price A: 6.547678467135405
Price B: 5.9272060658434915
Price A + B: 12.474884532978896
Price A + Price B: 12.474884532978898
Discrepancy: -1.7763568394002505e-15
Linearity holds: True

Pair: 2
Price A: 1.418219712533602
Price B: 4.935810293929461
Price A + B: 6.354030006463065
Price A + Price B: 6.354030006463063
Discrepancy: 1.7763568394002505e-15
Linearity holds: True

Pair: 3
Price A: 6.048086830681007
Price B: 5.365270081959249
Price A + B: 11.413356912640257
Price A + Price B: 11.413356912640257
Discrepancy: 0.0
Linearity holds: True

Pair: 4
Price A: 4.185277479337586
Price B: 1.661569934499096
Price A + B: 5.8468474138

### Interpretation of the 100-Claim Linearity Stress Test

The stress test generated

$$
100
$$

two-state contingent claims using the fixed random seed

$$
\boxed{42}.
$$

Python confirms that the generated claim array has shape

$$
\boxed{(100,2)},
$$

meaning that each of the 100 claims contains one payoff for each of the two
future states.

---

#### 1. Reproducibility

The first five generated claims are

$$
\begin{pmatrix}
3.74540119 & 9.50714306\\
7.31993942 & 5.98658484\\
1.56018640 & 1.55994520\\
0.58083612 & 8.66176146\\
6.01115012 & 7.08072578
\end{pmatrix}.
$$

Because the random seed is fixed, the same claim sample can be reproduced in a
future execution of the notebook.

This makes the stress test deterministic and auditable.

---

#### 2. Pairing Structure

The 100 claims were grouped sequentially into

$$
\boxed{50}
$$

non-overlapping claim pairs.

For each pair $(A_1,B_1)$, the reusable function independently computed

$$
p(A_1),
$$

$$
p(B_1),
$$

and

$$
p(A_1+B_1).
$$

The test then evaluated whether

$$
p(A_1+B_1)
\approx
p(A_1)+p(B_1)
$$

within floating-point tolerance.

Python reports

`Number of claim pairs tested: 50`

and

`All pairs satisfy linearity: True`.

Therefore,

$$
\boxed{
\text{all 50 pairwise linearity tests passed}.
}
$$

---

#### 3. Numerical Discrepancies

For each pair, the discrepancy was defined as

$$
\delta
=
p(A_1+B_1)
-
\left[p(A_1)+p(B_1)\right].
$$

The observed discrepancies were either zero or extremely small quantities such
as

$$
\pm1.7763568394002505\times10^{-15}
$$

and

$$
-8.881784197001252\times10^{-16}.
$$

The largest absolute discrepancy across all 50 tests was

$$
\boxed{
3.552713678800501\times10^{-15}.
}
$$

This is numerically negligible and is consistent with finite-precision
floating-point arithmetic.

It does not represent an economically meaningful pricing inconsistency.

---

#### 4. Example: First Pair

For the first pair, Python reports

$$
p(A_1)
\approx
6.5476784671
$$

and

$$
p(B_1)
\approx
5.9272060658.
$$

Therefore,

$$
p(A_1)+p(B_1)
\approx
12.4748845330.
$$

The combined claim is priced independently as

$$
p(A_1+B_1)
\approx
12.4748845330.
$$

The discrepancy is only

$$
-1.7763568394002505\times10^{-15}.
$$

Thus,

$$
\boxed{
p(A_1+B_1)
\approx
p(A_1)+p(B_1)
}
$$

to machine precision.

---

#### 5. Example: Exact Floating-Point Agreement

Some pairs produced a discrepancy of exactly

$$
0
$$

in the stored floating-point calculation.

For example, the third displayed pair satisfies

$$
p(A_1+B_1)
=
11.413356912640257
$$

and

$$
p(A_1)+p(B_1)
=
11.413356912640257.
$$

This does not make that pair theoretically more linear than the others.

All pairs satisfy the same mathematical property.

The difference is only in how the finite-precision arithmetic happened to
round.

---

#### 6. What the Stress Test Demonstrates

Part 8 established theoretically that replication pricing is linear because

$$
M(\phi^A+\phi^B)
=
M\phi^A+M\phi^B
$$

and

$$
M_0^\top(\phi^A+\phi^B)
=
M_0^\top\phi^A
+
M_0^\top\phi^B.
$$

The stress test does not create that theoretical result.

Instead, it verifies that the implementation respects it across many different
claims.

The experiment therefore provides evidence that the reusable pricing function
is behaving consistently with the underlying linear financial model.

---

#### 7. Why Every Random Claim Can Be Priced Here

The original market satisfies

$$
\operatorname{rank}(M)=2.
$$

Because the payoff space is also two-dimensional,

$$
\operatorname{col}(M)=\mathbb{R}^2.
$$

Therefore every generated claim

$$
C_1\in\mathbb{R}^2
$$

lies in the attainable payoff space.

That is why the function can solve every one of the randomly generated
replication problems.

This contrasts with the singular-market experiment, where the attainable
payoff space collapsed to a one-dimensional subset of

$$
\mathbb{R}^2.
$$

---

#### 8. Central Result

The stress test confirms that, for the complete two-state market,

$$
\boxed{
p(A_1+B_1)
=
p(A_1)+p(B_1)
}
$$

is preserved numerically across all 50 tested claim pairs.

The largest observed deviation was only

$$
\boxed{
3.55\times10^{-15},
}
$$

which is attributable to floating-point representation.

Thus,

$$
\boxed{
\text{the implementation is consistent with linear replication pricing}.
}
$$

The next section examines a subtler numerical problem: a payoff matrix that is
technically invertible but nearly singular.

## Part 10 — Near-Singularity and Numerical Conditioning

A singular payoff matrix has exactly dependent payoff columns.

A **near-singular** payoff matrix is different: its payoff columns are not
exactly dependent, so the matrix remains invertible, but they are almost
redundant.

Consider

$$
M_{\mathrm{near}}
=
\begin{pmatrix}
11&22\\
11&22.01
\end{pmatrix}.
$$

The first payoff column is

$$
\begin{pmatrix}
11\\
11
\end{pmatrix},
$$

while the second is

$$
\begin{pmatrix}
22\\
22.01
\end{pmatrix}.
$$

The second column is almost

$$
2
\begin{pmatrix}
11\\
11
\end{pmatrix}
=
\begin{pmatrix}
22\\
22
\end{pmatrix}.
$$

Thus, the two traded payoff directions are nearly linearly dependent.

---

### 1. The Matrix Is Still Invertible

Its determinant is

$$
\det(M_{\mathrm{near}})
=
11(22.01)-22(11).
$$

Therefore,

$$
\det(M_{\mathrm{near}})
=
242.11-242
=
0.11.
$$

Hence,

$$
\boxed{
\det(M_{\mathrm{near}})=0.11\neq0.
}
$$

The matrix is therefore mathematically invertible.

However, the determinant alone is not a scale-independent measure of numerical
stability.

To diagnose near-singularity more directly, the experiment also calculates the
matrix condition number.

---

### 2. Condition Number

The condition number measures how sensitive a linear system can be to small
changes in its inputs.

For the Euclidean norm,

$$
\kappa(M)
=
\frac{\sigma_{\max}(M)}
{\sigma_{\min}(M)},
$$

where

$$
\sigma_{\max}
$$

and

$$
\sigma_{\min}
$$

are the largest and smallest singular values.

A condition number close to $1$ indicates a well-conditioned system.

A very large condition number indicates that small perturbations can be
amplified substantially in the computed solution.

Thus,

$$
\boxed{
\text{large }\kappa(M)
\Longrightarrow
\text{greater numerical sensitivity}.
}
$$

The original payoff matrix and the near-singular matrix will be compared
directly.

---

### 3. Exact Replication Requires Large Offsetting Holdings

The target claim remains

$$
C_1=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
$$

Replication under the near-singular matrix requires

$$
M_{\mathrm{near}}\phi=C_1.
$$

Thus,

$$
11\phi_1+22\phi_2=5
$$

and

$$
11\phi_1+22.01\phi_2=0.
$$

Subtracting the first equation from the second gives

$$
0.01\phi_2=-5.
$$

Therefore,

$$
\boxed{
\phi_2=-500.
}
$$

Substituting this into the first equation,

$$
11\phi_1+22(-500)=5.
$$

Hence,

$$
11\phi_1-11000=5,
$$

so

$$
11\phi_1=11005.
$$

Therefore,

$$
\boxed{
\phi_1
=
\frac{11005}{11}
\approx
1000.454545.
}
$$

The exact hedge therefore uses approximately

$$
\boxed{
(1000.45,-500).
}
$$

These positions are dramatically larger than those required in the original
market.

The reason is that two nearly redundant payoff patterns must be combined in
large opposite positions to create the relatively small state difference

$$
(5,0).
$$

---

### 4. Why Large Offsetting Positions Matter

The two portfolio components generate very large cash flows that almost cancel.

This means that small numerical or economic perturbations can have a much
larger effect on the inferred holdings than in a well-conditioned system.

Near-singularity is therefore not equivalent to singularity:

- the system still has a unique exact solution;
- but that solution may be numerically sensitive.

---

### 5. Comparing Two Numerical Methods

The experiment compares

`np.linalg.solve(...)`

with

`np.linalg.lstsq(...)`.

For a square, full-rank matrix, both methods should recover essentially the same
replicating portfolio.

`np.linalg.solve` directly solves

$$
M_{\mathrm{near}}\phi=C_1.
$$

`np.linalg.lstsq` solves the least-squares problem

$$
\min_\phi
\|M_{\mathrm{near}}\phi-C_1\|_2.
$$

Because the matrix here is still full rank and the claim is exactly attainable,
the minimum residual should also be essentially zero.

The comparison will examine

- the determinant;
- matrix rank;
- singular values;
- condition number;
- holdings from both numerical methods;
- replication residual norms;
- agreement between the two computed portfolios.

In [10]:
M_near = np.array([
    [11.0, 22.0],
    [11.0, 22.01],
])

det_near = np.linalg.det(M_near)
rank_near = np.linalg.matrix_rank(M_near)

condition_original = np.linalg.cond(M)
condition_near = np.linalg.cond(M_near)

singular_values_near = np.linalg.svd(
    M_near,
    compute_uv=False,
)

phi_solve = np.linalg.solve(
    M_near,
    C1,
)

replicated_solve = M_near @ phi_solve
residual_solve = replicated_solve - C1
residual_norm_solve = np.linalg.norm(
    residual_solve
)

phi_lstsq, _, rank_lstsq, singular_values_lstsq = (
    np.linalg.lstsq(
        M_near,
        C1,
        rcond=None,
    )
)

replicated_lstsq = M_near @ phi_lstsq
residual_lstsq = replicated_lstsq - C1
residual_norm_lstsq = np.linalg.norm(
    residual_lstsq
)

holdings_agree = np.allclose(
    phi_solve,
    phi_lstsq,
)

print("Near-singular payoff matrix:")
print(M_near)

print()
print("Determinant:", det_near)
print("Rank:", rank_near)

print()
print("Original matrix condition number:", condition_original)
print("Near-singular condition number:", condition_near)
print("Near-singular singular values:", singular_values_near)

print()
print("Holdings from np.linalg.solve:")
print(phi_solve)

print("Replicated payoff from solve:")
print(replicated_solve)

print("Solve residual:")
print(residual_solve)

print("Solve residual norm:")
print(residual_norm_solve)

print()
print("Holdings from np.linalg.lstsq:")
print(phi_lstsq)

print("Least-squares rank:")
print(rank_lstsq)

print("Least-squares singular values:")
print(singular_values_lstsq)

print("Replicated payoff from lstsq:")
print(replicated_lstsq)

print("Least-squares residual:")
print(residual_lstsq)

print("Least-squares residual norm:")
print(residual_norm_lstsq)

print()
print(
    "solve and lstsq holdings agree:",
    holdings_agree,
)

Near-singular payoff matrix:
[[11.   22.  ]
 [11.   22.01]]

Determinant: 0.11000000000001717
Rank: 2

Original matrix condition number: 3.7777138829347794
Near-singular condition number: 11004.000818211023
Near-singular singular values: [3.47913795e+01 3.16170274e-03]

Holdings from np.linalg.solve:
[1000.45454545 -500.        ]
Replicated payoff from solve:
[ 5.00000000e+00 -1.13686838e-13]
Solve residual:
[-1.13686838e-13 -1.13686838e-13]
Solve residual norm:
1.6077746776921858e-13

Holdings from np.linalg.lstsq:
[1000.45454545 -500.        ]
Least-squares rank:
2
Least-squares singular values:
[3.47913795e+01 3.16170274e-03]
Replicated payoff from lstsq:
[ 5.00000000e+00 -2.27373675e-13]
Least-squares residual:
[-2.27373675e-13 -2.27373675e-13]
Least-squares residual norm:
3.2155493553843715e-13

solve and lstsq holdings agree: True


### Interpretation of Near-Singularity and Numerical Conditioning

The near-singular payoff matrix is

$$
M_{\mathrm{near}}
=
\begin{pmatrix}
11&22\\
11&22.01
\end{pmatrix}.
$$

Python reports

$$
\det(M_{\mathrm{near}})
\approx
0.11
$$

and

$$
\operatorname{rank}(M_{\mathrm{near}})=2.
$$

Therefore,

$$
\boxed{
M_{\mathrm{near}}
\text{ is still mathematically invertible}.
}
$$

This distinguishes the near-singular experiment from the exactly singular
matrix examined earlier.

---

#### 1. The Determinant Is Nonzero

The theoretical determinant is

$$
\det(M_{\mathrm{near}})
=
11(22.01)-22(11)
=
0.11.
$$

Python prints

$$
0.11000000000001717.
$$

The slight difference from the exact decimal value $0.11$ is a floating-point
representation effect.

The important structural fact is

$$
\boxed{
\det(M_{\mathrm{near}})\neq0.
}
$$

Thus, unlike the singular matrix from Part 7, this matrix has an inverse and
supports a unique solution to the square replication system.

---

#### 2. Full Rank Does Not Mean Well Conditioned

Python also reports

$$
\operatorname{rank}(M_{\mathrm{near}})=2.
$$

Therefore the two payoff columns are still linearly independent.

However, they are almost dependent.

The original market has condition number

$$
\kappa(M)
\approx
3.78,
$$

while the near-singular matrix has condition number

$$
\boxed{
\kappa(M_{\mathrm{near}})
\approx
11004.
}
$$

Thus,

$$
\frac{\kappa(M_{\mathrm{near}})}
{\kappa(M)}
$$

is extremely large.

The near-singular system is therefore much more sensitive to numerical or
input perturbations than the original market.

---

#### 3. Singular Values Explain the Large Condition Number

The singular values of the near-singular matrix are approximately

$$
\sigma_{\max}
=
34.7913795
$$

and

$$
\sigma_{\min}
=
0.00316170.
$$

The condition number is

$$
\kappa(M_{\mathrm{near}})
=
\frac{\sigma_{\max}}
{\sigma_{\min}}.
$$

Because the smallest singular value is very close to zero,

$$
\boxed{
\sigma_{\min}\approx0,
}
$$

the matrix lies close to rank deficiency.

That is the numerical signature of near-singularity.

The matrix has not yet lost rank, but one payoff direction is almost redundant
with the other.

---

#### 4. Large Offsetting Replicating Positions

Using `np.linalg.solve`, Python obtains

$$
\phi_{\mathrm{solve}}
\approx
\begin{pmatrix}
1000.45454545\\
-500
\end{pmatrix}.
$$

This agrees with the hand derivation.

The holdings are extremely large relative to the target claim payoff

$$
C_1=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
$$

The reason is that the traded payoff vectors are almost parallel.

To create a payoff difference of only $5$ between the two states, the
replication strategy must combine very large positions with opposite signs.

Thus,

$$
\boxed{
\text{near-redundant assets}
\rightarrow
\text{large offsetting holdings}.
}
$$

This is an important financial consequence of ill-conditioning.

---

#### 5. Why the Replication Still Works

Despite the large portfolio positions, the direct solver produces

$$
M_{\mathrm{near}}\phi_{\mathrm{solve}}
\approx
\begin{pmatrix}
5\\
0
\end{pmatrix}.
$$

The residual norm is only

$$
\|e_{\mathrm{solve}}\|_2
\approx
1.61\times10^{-13}.
$$

Therefore,

$$
\boxed{
\|e_{\mathrm{solve}}\|_2\approx0.
}
$$

The target claim is still replicated to machine precision.

The small residual is numerical rounding error rather than a material payoff
mismatch.

---

#### 6. Least-Squares Solution

The least-squares method returns

$$
\phi_{\mathrm{lstsq}}
\approx
\begin{pmatrix}
1000.45454545\\
-500
\end{pmatrix}.
$$

This is essentially the same portfolio as the direct solution.

Python confirms

`solve and lstsq holdings agree: True`.

The least-squares method also reports rank

$$
2,
$$

confirming again that the matrix remains full rank.

Its singular values match those computed directly from the singular value
decomposition.

---

#### 7. Least-Squares Residual

The least-squares replication residual norm is approximately

$$
\|e_{\mathrm{lstsq}}\|_2
\approx
3.22\times10^{-13}.
$$

This is also effectively zero at machine precision.

Thus, both methods produce an essentially exact hedge:

$$
\boxed{
M_{\mathrm{near}}\phi
\approx
C_1.
}
$$

The fact that one residual is slightly smaller than the other in this particular
floating-point calculation should not be interpreted as an economically
meaningful difference.

Both are negligible relative to the payoff scale.

---

#### 8. Why `solve` and `lstsq` Agree Here

`np.linalg.solve` is designed to solve a square system

$$
M\phi=C_1
$$

directly when the matrix is nonsingular.

`np.linalg.lstsq` instead solves

$$
\min_\phi
\|M\phi-C_1\|_2.
$$

In this experiment,

$$
M_{\mathrm{near}}
$$

is square, full rank, and the claim is exactly attainable.

Therefore, the minimum achievable residual is essentially zero, and the
least-squares solution coincides with the unique direct solution.

Hence,

$$
\boxed{
\phi_{\mathrm{solve}}
\approx
\phi_{\mathrm{lstsq}}.
}
$$

The two functions approach the numerical problem differently, but under these
conditions they recover the same financial hedge.

---

#### 9. Singularity, Near-Singularity, and Ill-Conditioning

These concepts should be distinguished carefully.

A **singular matrix** has exact rank deficiency:

$$
\det(M)=0.
$$

It has no inverse, and a square system may have either no solution or multiple
solutions depending on the target claim.

A **near-singular matrix** is still full rank:

$$
\det(M)\neq0,
$$

but it lies close to a singular matrix because its payoff directions are almost
linearly dependent.

An **ill-conditioned matrix** is numerically sensitive: small perturbations in
the inputs can produce comparatively large changes in the solution.

Near-singularity often produces ill-conditioning because the smallest singular
value approaches zero.

For this experiment,

$$
\boxed{
M_{\mathrm{near}}
\text{ is nonsingular but severely ill-conditioned}.
}
$$

---

#### 10. Financial Interpretation of Ill-Conditioning

The numerical instability has a financial interpretation.

The two traded assets provide almost the same state-payoff pattern.

Therefore, the market has very little effective separation between the two
payoff directions.

To manufacture a claim that differs meaningfully across states, the replication
strategy must take very large positions that nearly cancel one another.

That means that small changes in

- traded-asset payoffs;
- claim payoffs;
- measurement precision;

can lead to much larger changes in the inferred portfolio holdings.

Thus,

$$
\boxed{
\text{mathematical completeness}
\not\Rightarrow
\text{numerical robustness}.
}
$$

The market is complete in the strict linear-algebra sense because the matrix
has full rank, but replication can still be numerically fragile.

---

#### 11. Contrast With the Three Market Structures

The notebook has now examined three distinct cases.

For the original market,

$$
\operatorname{rank}(M)=2
$$

and the condition number is modest:

$$
\kappa(M)\approx3.78.
$$

Replication is unique and numerically stable.

For the singular market,

$$
\operatorname{rank}(M_{\mathrm{sing}})=1.
$$

The payoff space collapses to a lower-dimensional subspace and unique
replication fails.

For the near-singular market,

$$
\operatorname{rank}(M_{\mathrm{near}})=2,
$$

but

$$
\kappa(M_{\mathrm{near}})
\approx11004.
$$

Unique exact replication still exists, but it requires very large offsetting
positions and is numerically sensitive.

The comparison is therefore

$$
\boxed{
\begin{array}{c|c|c|c}
\text{Case}
&
\text{Rank}
&
\text{Unique exact solution}
&
\text{Numerical behavior}
\\
\hline
\text{Original}
&
2
&
\text{Yes}
&
\text{Well conditioned}
\\
\text{Singular}
&
1
&
\text{Not generally}
&
\text{Rank deficient}
\\
\text{Near-singular}
&
2
&
\text{Yes}
&
\text{Ill conditioned}
\end{array}
}
$$

---

#### 12. Central Result

The near-singular experiment demonstrates that checking only whether a matrix
is invertible is not enough to understand the numerical quality of a
replication problem.

For

$$
M_{\mathrm{near}},
$$

the determinant is nonzero and the rank is full, so a unique replication
portfolio exists.

However,

$$
\kappa(M_{\mathrm{near}})
\approx11004
$$

and the hedge requires positions near

$$
\boxed{
(1000.45,-500).
}
$$

Therefore,

$$
\boxed{
\text{unique replication can exist while still being numerically fragile}.
}
$$

This completes the capstone's main replication, pricing, singularity,
linearity, and numerical-robustness experiments.

## Part 11 — Financial Conclusions

This capstone developed replication pricing in a complete two-state market and
then tested how the same framework behaves when the payoff system becomes
singular or nearly singular.

The central financial relationship is

$$
\boxed{
\text{traded-asset payoffs}
\rightarrow
\text{replication}
\rightarrow
\text{no-arbitrage price}.
}
$$

The numerical experiments also show that the existence of a mathematical
solution does not by itself guarantee numerical robustness.

---

### 1. Completeness Comes From Spanning the Payoff Space

The original traded-asset payoff matrix is

$$
M=
\begin{pmatrix}
11&20\\
11&5
\end{pmatrix}.
$$

Its determinant is nonzero and its rank is

$$
\operatorname{rank}(M)=2.
$$

Because the two-state contingent-claim space is

$$
\mathbb{R}^2,
$$

the two independent traded payoff vectors span the entire payoff space:

$$
\boxed{
\operatorname{col}(M)=\mathbb{R}^2.
}
$$

Therefore every two-state contingent claim is attainable using the traded
bond and stock under the unrestricted-holdings framework of this model.

This is the linear-algebra meaning of market completeness in the present
setting.

---

### 2. Exact Replication Determines the Hedge

For the call claim with strike

$$
K=15,
$$

the future payoff vector is

$$
C_1=
\begin{pmatrix}
5\\
0
\end{pmatrix}.
$$

Solving

$$
M\phi=C_1
$$

gives the unique replicating portfolio

$$
\boxed{
\phi=
\begin{pmatrix}
-\frac5{33}\\
\frac13
\end{pmatrix}.
}
$$

Thus, the hedge consists of

- a short position of $\frac5{33}$ units in the bond;
- a long position of $\frac13$ of a stock unit.

The portfolio reproduces the claim payoff state by state:

$$
\boxed{
M\phi=C_1.
}
$$

The negative bond holding represents borrowing through the bond position.

---

### 3. Replication Determines the Arbitrage-Free Price

The current traded-asset price vector is

$$
M_0=
\begin{pmatrix}
10\\
10
\end{pmatrix}.
$$

The current cost of the replicating portfolio is

$$
C_0=M_0^\top\phi.
$$

Substituting the exact holdings gives

$$
C_0
=
10\left(-\frac5{33}\right)
+
10\left(\frac13\right)
=
\frac{20}{11}.
$$

Therefore,

$$
\boxed{
C_0=
\frac{20}{11}
\approx1.81818.
}
$$

This is the claim's no-arbitrage price because the claim and its replicating
portfolio have identical future payoffs.

If their current prices differed, one could buy the cheaper position and sell
the more expensive position while their future cash flows offset exactly.

Thus,

$$
\boxed{
\text{exact replication}
+
\text{no arbitrage}
\Longrightarrow
\text{replication price}.
}
$$

No physical state probabilities were required for this pricing argument.

---

### 4. The Reusable Function Encodes the Financial Logic

The function

`price_by_replication(M, M0, C1)`

implements the sequence

$$
M\phi=C_1,
$$

followed by the residual check

$$
\|M\phi-C_1\|_2\approx0,
$$

and then the price calculation

$$
C_0=M_0^\top\phi.
$$

This ordering matters.

The function does not assign a replication price until the future payoff match
has first been verified numerically.

The function therefore produces both

$$
\boxed{\text{the hedge}}
$$

and

$$
\boxed{\text{the price}}.
$$

---

### 5. Singularity Reveals Incompleteness

The deliberately singular payoff matrix

$$
M_{\mathrm{sing}}
=
\begin{pmatrix}
11&22\\
11&22
\end{pmatrix}
$$

has rank

$$
1.
$$

Its second payoff column is exactly twice the first, so the traded assets span
only a one-dimensional payoff space.

Every attainable payoff must have the form

$$
(x,x).
$$

The target claim

$$
C_1=(5,0)
$$

does not lie in that subspace.

Therefore,

$$
\boxed{
C_1\notin\operatorname{col}(M_{\mathrm{sing}})
}
$$

and no exact replication exists for that claim.

The singularity experiment demonstrates that the failure of the numerical
solver reflects a financial failure of the payoff system to span the desired
claim.

---

### 6. Singularity Does Not Always Mean No Solution

A singular matrix does not imply that every claim is unattainable.

Claims lying inside

$$
\operatorname{col}(M_{\mathrm{sing}})
$$

can still be generated.

However, because the payoff columns are redundant, such claims need not have a
unique replicating portfolio.

Thus, rank deficiency can lead to

- no solution for claims outside the attainable payoff space; or
- multiple portfolio solutions for claims inside the attainable payoff space.

The reusable pricing function deliberately rejects singular square systems
because it is designed for unique exact replication.

---

### 7. Replication Pricing Is Linear

For two attainable claims $A_1$ and $B_1$,

$$
M\phi^A=A_1
$$

and

$$
M\phi^B=B_1.
$$

By linearity,

$$
M(\phi^A+\phi^B)=A_1+B_1.
$$

Therefore,

$$
\phi^{A+B}
=
\phi^A+\phi^B.
$$

Because current portfolio cost is also linear,

$$
\boxed{
p(A_1+B_1)
=
p(A_1)+p(B_1).
}
$$

The explicit two-claim experiment confirmed both portfolio and pricing
additivity.

---

### 8. The 100-Claim Stress Test Confirms the Implementation

Using random seed

$$
42,
$$

the notebook generated 100 reproducible two-state claims and grouped them into
50 pairs.

Every pair satisfied

$$
p(A_1+B_1)
\approx
p(A_1)+p(B_1)
$$

within floating-point tolerance.

Python reported

$$
\boxed{
\text{All pairs satisfy linearity: True}.
}
$$

The largest absolute discrepancy was approximately

$$
3.55\times10^{-15}.
$$

This is consistent with floating-point rounding and is economically negligible.

The stress test therefore supports the conclusion that the reusable
implementation preserves the theoretical linear pricing structure.

---

### 9. Near-Singularity Is Different From Singularity

The near-singular matrix

$$
M_{\mathrm{near}}
=
\begin{pmatrix}
11&22\\
11&22.01
\end{pmatrix}
$$

still has

$$
\operatorname{rank}(M_{\mathrm{near}})=2
$$

and

$$
\det(M_{\mathrm{near}})\neq0.
$$

Therefore, a unique exact replication portfolio still exists.

However, the matrix condition number is approximately

$$
\boxed{
\kappa(M_{\mathrm{near}})
\approx11004,
}
$$

compared with only about

$$
3.78
$$

for the original market.

The near-singular system is therefore dramatically more ill-conditioned.

---

### 10. Completeness Does Not Guarantee Numerical Robustness

For the same target claim,

$$
C_1=(5,0),
$$

the near-singular system requires approximately

$$
\boxed{
\phi
=
\begin{pmatrix}
1000.45\\
-500
\end{pmatrix}.
}
$$

The claim payoff is small relative to the size of these holdings.

The hedge works because very large opposing payoff contributions almost cancel.

Both

`np.linalg.solve`

and

`np.linalg.lstsq`

recover essentially the same hedge, with residual norms near machine
precision.

Nevertheless, the large condition number shows that the solution is much more
sensitive to perturbations than the original replication problem.

Thus,

$$
\boxed{
\text{full rank}
\not\Rightarrow
\text{good numerical conditioning}.
}
$$

A market can be mathematically complete while its replication problem remains
numerically fragile.

---

### 11. Overall Comparison

The three payoff systems illustrate distinct cases:

$$
\begin{array}{c|c|c|c}
\text{Payoff system}
&
\text{Rank}
&
\text{Replication}
&
\text{Numerical character}
\\
\hline
\text{Original market}
&
2
&
\text{unique for every two-state claim}
&
\text{well conditioned}
\\
\text{Singular market}
&
1
&
\text{not uniquely available in general}
&
\text{rank deficient}
\\
\text{Near-singular market}
&
2
&
\text{unique}
&
\text{ill conditioned}
\end{array}
$$

This comparison connects the financial concepts of attainability and
completeness with the numerical concepts of rank and conditioning.

---

### 12. Final Financial Takeaway

The Chapter 2 framework can be summarized as

$$
\boxed{
\text{independent traded payoffs}
\rightarrow
\text{complete payoff span}
\rightarrow
\text{unique replication}
\rightarrow
\text{unique no-arbitrage price}.
}
$$

At the same time, the numerical experiments add an important qualification:

$$
\boxed{
\text{existence of a unique hedge}
\neq
\text{numerical stability of that hedge}.
}
$$

Replication pricing is therefore simultaneously a financial argument and a
linear-algebra problem.

Understanding both dimensions is essential for interpreting the output of the
Python implementation rather than treating the numerical solver as a black box.

## Part 12 — Validation and Reproducibility Checks

The final section performs explicit validation checks on the central results of
the capstone.

The purpose is not to replace the financial derivations developed earlier.

Instead, the validation layer checks that the completed Python implementation
remains internally consistent and reproducible.

The tests cover five areas:

1. the structure and completeness of the original market;
2. exact replication and the arbitrage-free price;
3. controlled rejection of the singular market;
4. the 100-claim linearity stress test;
5. the near-singular numerical experiment.

A check is reported as

`PASS`

only when its stated numerical or logical condition is satisfied.

For quantities affected by floating-point arithmetic, comparisons use numerical
tolerances rather than requiring identical decimal representations.

After all individual checks are evaluated, the notebook reports an overall
validation result.

A failed check raises an error deliberately so that a later top-to-bottom
execution cannot silently appear successful when a required result has broken.

The completed notebook should therefore finish with

$$
\boxed{
\text{ALL VALIDATION CHECKS PASSED}
}
$$

before it is considered ready for the final fresh-runtime test.

In [11]:
validation_results = []


def validate(name, condition):
    condition = bool(condition)
    validation_results.append((name, condition))

    status = "PASS" if condition else "FAIL"
    print(f"{status}: {name}")


# ------------------------------------------------------------
# 1. Original market structure
# ------------------------------------------------------------

validate(
    "Original payoff matrix has shape (2, 2)",
    M.shape == (2, 2),
)

validate(
    "Original payoff matrix determinant is -165",
    np.isclose(np.linalg.det(M), -165.0),
)

validate(
    "Original payoff matrix has rank 2",
    np.linalg.matrix_rank(M) == 2,
)

validate(
    "Call payoff is C1 = (5, 0)",
    np.allclose(
        C1,
        np.array([5.0, 0.0]),
    ),
)


# ------------------------------------------------------------
# 2. Exact replication and price
# ------------------------------------------------------------

expected_phi = np.array([
    -5.0 / 33.0,
    1.0 / 3.0,
])

expected_C0 = 20.0 / 11.0

validation_phi, validation_C0 = price_by_replication(
    M,
    M0,
    C1,
)

validation_payoff = M @ validation_phi
validation_residual = validation_payoff - C1
validation_residual_norm = np.linalg.norm(
    validation_residual
)

validate(
    "Replication holdings match exact solution",
    np.allclose(
        validation_phi,
        expected_phi,
    ),
)

validate(
    "Replicating portfolio reproduces the claim",
    np.allclose(
        validation_payoff,
        C1,
    ),
)

validate(
    "Replication residual norm is below 1e-10",
    validation_residual_norm < 1e-10,
)

validate(
    "Replication price equals 20/11",
    np.isclose(
        validation_C0,
        expected_C0,
    ),
)


# ------------------------------------------------------------
# 3. Singular-market checks
# ------------------------------------------------------------

validate(
    "Singular payoff matrix has rank 1",
    np.linalg.matrix_rank(M_singular) == 1,
)

rank_singular_validation = np.linalg.matrix_rank(
    M_singular
)

rank_augmented_validation = np.linalg.matrix_rank(
    np.column_stack(
        (M_singular, C1)
    )
)

validate(
    "Selected claim is unattainable in singular market",
    rank_singular_validation
    < rank_augmented_validation,
)

singular_rejected_validation = False

try:
    price_by_replication(
        M_singular,
        M0,
        C1,
    )

except ValueError:
    singular_rejected_validation = True

validate(
    "Pricing function rejects singular payoff matrix",
    singular_rejected_validation,
)


# ------------------------------------------------------------
# 4. 100-claim linearity stress test
# ------------------------------------------------------------

validate(
    "Stress test generated 100 two-state claims",
    random_claims.shape == (100, 2),
)

validate(
    "Stress test evaluated 50 claim pairs",
    len(linearity_records) == 50,
)

validate(
    "All 50 claim pairs satisfy pricing linearity",
    all_pairs_linear,
)

validate(
    "Maximum linearity discrepancy is below 1e-12",
    max_absolute_discrepancy < 1e-12,
)


# ------------------------------------------------------------
# 5. Near-singular numerical checks
# ------------------------------------------------------------

validate(
    "Near-singular payoff matrix remains full rank",
    np.linalg.matrix_rank(M_near) == 2,
)

validate(
    "Near-singular matrix is much more ill-conditioned",
    condition_near
    > 1000.0 * condition_original,
)

validate(
    "solve and lstsq produce matching holdings",
    np.allclose(
        phi_solve,
        phi_lstsq,
    ),
)

validate(
    "solve residual norm is below 1e-10",
    residual_norm_solve < 1e-10,
)

validate(
    "lstsq residual norm is below 1e-10",
    residual_norm_lstsq < 1e-10,
)


# ------------------------------------------------------------
# Overall validation result
# ------------------------------------------------------------

checks_passed = sum(
    passed
    for _, passed in validation_results
)

total_checks = len(validation_results)

all_validation_passed = all(
    passed
    for _, passed in validation_results
)

print()
print(
    f"Validation summary: "
    f"{checks_passed}/{total_checks} checks passed."
)

if all_validation_passed:
    print("ALL VALIDATION CHECKS PASSED")

else:
    failed_checks = [
        name
        for name, passed in validation_results
        if not passed
    ]

    raise AssertionError(
        "Validation failed: "
        + "; ".join(failed_checks)
    )

PASS: Original payoff matrix has shape (2, 2)
PASS: Original payoff matrix determinant is -165
PASS: Original payoff matrix has rank 2
PASS: Call payoff is C1 = (5, 0)
PASS: Replication holdings match exact solution
PASS: Replicating portfolio reproduces the claim
PASS: Replication residual norm is below 1e-10
PASS: Replication price equals 20/11
PASS: Singular payoff matrix has rank 1
PASS: Selected claim is unattainable in singular market
PASS: Pricing function rejects singular payoff matrix
PASS: Stress test generated 100 two-state claims
PASS: Stress test evaluated 50 claim pairs
PASS: All 50 claim pairs satisfy pricing linearity
PASS: Maximum linearity discrepancy is below 1e-12
PASS: Near-singular payoff matrix remains full rank
PASS: Near-singular matrix is much more ill-conditioned
PASS: solve and lstsq produce matching holdings
PASS: solve residual norm is below 1e-10
PASS: lstsq residual norm is below 1e-10

Validation summary: 20/20 checks passed.
ALL VALIDATION CHECKS PASSE